# Pipeline Integral de Preprocesamiento, Clasificación CNN y Explicabilidad (XAI) de Memes en Google Colab

### Aceleración por GPU en Google Colab (NVIDIA T4 / V100 / A100 / L4)
Este notebook está completamente adaptado y optimizado para ejecutarse en la nube de **Google Colab** con aceleración de GPU por hardware.

---

### Guía Rápida de Uso en Google Colab:
1. **Activar Entorno GPU**:
   * En el menú superior de Colab: **Entorno de ejecución** -> **Cambiar tipo de entorno de ejecución** -> Seleccionar **T4 GPU** (o superior).
2. **Carga de Datos desde GitHub**:
   * Sube tus carpetas con los memes (`animado/`, `animales/`, `personas/`) a tu repositorio de GitHub.
   * En la Celda 0 (abajo), introduce la URL de tu repositorio en la variable `GITHUB_REPO_URL`.
   * El script clonará automáticamente el repositorio y localizará las carpetas en cualquier subdirectorio.
   * *(Alternativa)*: También puedes subir un archivo `.zip` al panel izquierdo de Colab y descomprimirlo con `!unzip`.
3. **Pilares de la Arquitectura**:
   * **100% Código Propio**: Cero modelos preentrenados, cero transfer learning.
   * **Resolución Ultra-Nítida**: $384 \times 384$ px para preservar texto y detalles sutiles.
   * **Atajos Residuales y Global Average Pooling**: Prevención del desvanecimiento del gradiente y alta generalización.
   * **Explicabilidad Visual (XAI)**: Módulos de auditoría con **Grad-CAM** y **LIME** al final del pipeline.

In [ ]:
# ==============================================================================
# 0. CONFIGURACIÓN DEL ENTORNO GOOGLE COLAB Y CONEXIÓN CON GITHUB
# ==============================================================================
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🚀 Detectado entorno de ejecución: GOOGLE COLAB")
    
    # 1. Instalar librerías complementarias (LIME para explicabilidad XAI)
    print("📦 Verificando/instalando librería 'lime' para análisis XAI...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lime"], check=False)
    
    # 2. CONFIGURACIÓN DEL REPOSITORIO DE GITHUB
    # Sustituye esta URL por la de tu repositorio donde subiste las carpetas 'animado', 'animales' y 'personas':
    GITHUB_REPO_URL = "https://github.com/Luis7Cruz/dataset-memes-cnn.git"
    
    if GITHUB_REPO_URL and "TU_USUARIO" not in GITHUB_REPO_URL:
        repo_nombre = GITHUB_REPO_URL.split('/')[-1].replace('.git', '')
        if not Path(repo_nombre).exists() and not (Path('animado').exists() and Path('animales').exists()):
            print(f"📥 Clonando repositorio desde GitHub: {GITHUB_REPO_URL} ...")
            subprocess.run(["git", "clone", GITHUB_REPO_URL], check=False)
            if Path(repo_nombre).exists():
                print(f"📂 Repositorio clonado exitosamente en: ./{repo_nombre}")
    else:
        print("💡 CONSEJO: Asigna la URL de tu repositorio en 'GITHUB_REPO_URL'")
        print("   o sube tus carpetas/zip directamente al explorador de archivos de Colab (/content).")
else:
    print("💻 Entorno Local detectado.")

def resolver_directorio_base():
    """Localiza de forma automática y recursiva las carpetas del dataset en Colab o Local."""
    candidatos = [Path('.'), Path('..')] + list(Path('.').glob('*/')) + list(Path('.').glob('*/*/'))
    for cand in candidatos:
        if (cand / 'animado').is_dir() and (cand / 'animales').is_dir() and (cand / 'personas').is_dir():
            return cand.resolve()
    return Path('.').resolve()

BASE_DIR = resolver_directorio_base()
print(f"📁 Directorio base localizado del dataset: {BASE_DIR}")

In [ ]:
# 1. Importación de librerías esenciales y configuración de entorno
import os
import sys
import gc
import glob
import json
import time
import hashlib
import warnings
from pathlib import Path
from collections import Counter, defaultdict

# Control de logs
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# Computación numérica y datos
import numpy as np
import pandas as pd

# Procesamiento y visualización de imágenes
from PIL import Image, ImageOps, ImageFile
import cv2
import matplotlib.pyplot as plt
import seaborn as sns

# Habilitar carga de imágenes parcialmente truncadas en PIL para robustez
ImageFile.LOAD_TRUNCATED_IMAGES = True

# Framework principal: TensorFlow y Keras
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

# Evaluación y métricas de rendimiento
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split

# Configuración visual
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

warnings.filterwarnings('ignore')

print("Entorno configurado correctamente.")
print(f"Versión de Python: {sys.version.split()[0]}")
print(f"Versión de TensorFlow: {tf.__version__}")
print(f"Versión de OpenCV: {cv2.__version__}")
print(f"Versión de Pillow: {Image.__version__}")

# VERIFICACIÓN OBLIGATORIA DE ACELERACIÓN GPU
print("Dispositivos GPU detectados:", tf.config.list_physical_devices('GPU'))
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        detalles = tf.config.experimental.get_device_details(gpus[0])
        print(f"Aceleración activa en: {detalles.get('device_name', gpus[0].name)}")
    except Exception:
        print(f"Aceleración GPU activa: {gpus[0].name}")
else:
    print("⚠️ ATENCIÓN: No se detectó GPU. En Google Colab ve a 'Entorno de ejecución -> Cambiar tipo de entorno de ejecución -> T4 GPU'.")

## Fase 1: Auditoria Forense y Analisis Exploratorio de Datos (EDA)

Inspeccion de todas las carpetas (`animado`, `animales`, `personas`) para analizar:
* Integridad de archivos y deteccion de duplicados exactos mediante hash MD5.
* Formato MIME real vs extension declarada.
* Modos de color (RGB, RGBA, L, P, CMYK) y numero de canales reales.
* Deteccion de imagenes animadas o metadatos de orientacion EXIF.
* Rango numerico de resoluciones y relaciones de aspecto.

In [ ]:
# Escaneo forense de todo el dataset
BASE_DIR = resolver_directorio_base()
CATEGORIAS = ['animado', 'animales', 'personas']

registros_imagenes = []
errores_lectura = []
hashes_md5 = defaultdict(list)

for categoria in CATEGORIAS:
    dir_cat = BASE_DIR / categoria
    if not dir_cat.exists():
        print(f"Directorio no encontrado: {dir_cat}")
        continue
    
    archivos = [f for f in dir_cat.iterdir() if f.is_file()]
    
    for archivo in archivos:
        try:
            contenido = archivo.read_bytes()
            h_md5 = hashlib.md5(contenido).hexdigest()
            hashes_md5[h_md5].append(str(archivo))
            tam_kb = round(len(contenido) / 1024, 2)
            
            with Image.open(archivo) as img:
                w, h = img.size
                modo = img.mode
                formato_real = img.format
                es_animado = getattr(img, 'is_animated', False)
                n_frames = getattr(img, 'n_frames', 1)
                
                exif = img.getexif()
                orientacion_exif = exif.get(0x0112, 1) if exif else 1
                canales = len(img.getbands())
                aspect_ratio = round(w / h, 4)
                
                registros_imagenes.append({
                    'categoria': categoria,
                    'nombre_archivo': archivo.name,
                    'ruta_completa': str(archivo),
                    'extension': archivo.suffix.lower(),
                    'formato_real': formato_real,
                    'modo_color': modo,
                    'num_canales': canales,
                    'ancho': w,
                    'alto': h,
                    'megapixeles': round((w * h) / 1e6, 4),
                    'aspect_ratio': aspect_ratio,
                    'tamano_kb': tam_kb,
                    'es_animado': es_animado,
                    'n_frames': n_frames,
                    'orientacion_exif': orientacion_exif,
                    'hash_md5': h_md5
                })
        except Exception as e:
            errores_lectura.append({'archivo': str(archivo), 'error': str(e)})

df_memes = pd.DataFrame(registros_imagenes)
print(f"Total de imagenes analizadas con exito: {len(df_memes)}")
print(f"Total de errores de lectura: {len(errores_lectura)}")
if errores_lectura:
    print("Detalle de errores:", errores_lectura)

In [ ]:
# Resumen estadistico por categoria
resumen_clases = df_memes.groupby('categoria').agg(
    total_imagenes=('nombre_archivo', 'count'),
    ancho_min=('ancho', 'min'),
    ancho_mediana=('ancho', 'median'),
    ancho_max=('ancho', 'max'),
    alto_min=('alto', 'min'),
    alto_mediana=('alto', 'median'),
    alto_max=('alto', 'max'),
    ar_min=('aspect_ratio', 'min'),
    ar_mediana=('aspect_ratio', 'median'),
    ar_max=('aspect_ratio', 'max'),
    mp_mediana=('megapixeles', 'median')
)
print("Resumen estadistico por clase:")
print(resumen_clases.to_string())

print("\nDistribucion de modos de color detectados:")
print(df_memes['modo_color'].value_counts().to_string())

duplicados = {k: v for k, v in hashes_md5.items() if len(v) > 1}
print(f"\nGrupos de imagenes duplicadas (hash MD5 identico): {len(duplicados)}")

In [ ]:
# Visualizacion de distribuciones: Conteo, Dispersion de Resoluciones e Histograma de Aspect Ratio
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

palette = {'animado': '#4C72B0', 'animales': '#55A868', 'personas': '#C44E52'}

# 1. Distribucion de imagenes por clase
sns.countplot(data=df_memes, x='categoria', palette=palette, ax=axes[0])
axes[0].set_title('Distribucion de Imagenes por Clase', fontweight='bold')
axes[0].set_ylabel('Cantidad de Imagenes')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height())}', 
                     (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                     ha='center', va='center', fontsize=12, color='white', fontweight='bold')

# 2. Grafica de dispersion de dimensiones (Ancho vs Alto)
sns.scatterplot(data=df_memes, x='ancho', y='alto', hue='categoria', 
                style='categoria', s=80, alpha=0.8, palette=palette, ax=axes[1])
axes[1].plot([0, 1800], [0, 1800], 'r--', label='Relacion 1:1 (Cuadrada)')
axes[1].set_title('Dispersion de Dimensiones (Ancho vs Alto en px)', fontweight='bold')
axes[1].set_xlabel('Ancho (pixeles)')
axes[1].set_ylabel('Alto (pixeles)')
axes[1].legend()

# 3. Distribucion de Aspect Ratio (W / H)
sns.histplot(data=df_memes, x='aspect_ratio', hue='categoria', kde=True, 
             bins=25, palette=palette, ax=axes[2])
axes[2].axvline(1.0, color='red', linestyle='--', label='Cuadrado (1.0)')
axes[2].axvline(0.56, color='purple', linestyle=':', label='Vertical 9:16 (0.56)')
axes[2].axvline(1.78, color='green', linestyle=':', label='Panoramico 16:9 (1.78)')
axes[2].set_title('Distribucion de Relacion de Aspecto (W / H)', fontweight='bold')
axes[2].set_xlabel('Aspect Ratio (Ancho / Alto)')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Identificacion y visualizacion de casos limite (Outliers Extremos)
mas_vertical = df_memes.loc[df_memes['aspect_ratio'].idxmin()]
mas_horizontal = df_memes.loc[df_memes['aspect_ratio'].idxmax()]
menor_resolucion = df_memes.loc[df_memes['megapixeles'].idxmin()]
mayor_resolucion = df_memes.loc[df_memes['megapixeles'].idxmax()]

casos_limite = [
    ('Meme Mas Vertical (AR Minimo)', mas_vertical),
    ('Meme Mas Horizontal (AR Maximo)', mas_horizontal),
    ('Menor Resolucion (Baja Calidad)', menor_resolucion),
    ('Mayor Resolucion (Alta Calidad)', mayor_resolucion)
]

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for ax, (titulo, fila) in zip(axes, casos_limite):
    with Image.open(fila['ruta_completa']) as img_raw:
        # Redimensionar copia para visualizacion ligera sin sobrecargar memoria
        img_preview = img_raw.copy()
        img_preview.thumbnail((400, 400))
        ax.imshow(img_preview)
    ax.set_title(f"{titulo}\nClase: {fila['categoria']}\n{fila['ancho']}x{fila['alto']} px (AR: {fila['aspect_ratio']})", fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

## Fase 2: Analisis de Dimensiones y Deteccion de Sesgo de Clase con Boxplots

### La Pregunta Clave: ¿Puede la red aprender a clasificar por el tamaño de la caja?
En tareas de vision por computadora con datasets pequeños o recolectados de internet, existe el riesgo de **Shortcut Learning (Aprendizaje de Atajos Espurios)**:
* Si una categoria (por ejemplo, `animales`) siempre proviene de camaras o sitios web de mayor resolucion o con proporciones panoramicas, mientras que otra (`animado`) proviene de recortes pequeños o cuadrados.
* La red neuronal podria aprender a predecir la clase basandose simplemente en el tamaño de la imagen, el nivel de desenfoque o la presencia de barras de relleno, en lugar de identificar si hay un perro, una persona o un dibujo animado.

Para verificar si existe este riesgo en los datos crudos, analizamos la distribucion de las cuatro variables dimensionales clave mediante diagramas de caja (Boxplots).

In [ ]:
# Diagramas de caja (Boxplots) para evaluar el sesgo dimensional por clase
fig, axes = plt.subplots(2, 2, figsize=(15, 11))

palette = {'animado': '#4C72B0', 'animales': '#55A868', 'personas': '#C44E52'}

# 1. Boxplot de Ancho
sns.boxplot(data=df_memes, x='categoria', y='ancho', palette=palette, ax=axes[0, 0])
axes[0, 0].set_title('Distribucion de Ancho (pixeles) por Clase', fontweight='bold')
axes[0, 0].set_xlabel('Clase')
axes[0, 0].set_ylabel('Ancho (px)')

# 2. Boxplot de Alto
sns.boxplot(data=df_memes, x='categoria', y='alto', palette=palette, ax=axes[0, 1])
axes[0, 1].set_title('Distribucion de Alto (pixeles) por Clase', fontweight='bold')
axes[0, 1].set_xlabel('Clase')
axes[0, 1].set_ylabel('Alto (px)')

# 3. Boxplot de Relacion de Aspecto (W / H)
sns.boxplot(data=df_memes, x='categoria', y='aspect_ratio', palette=palette, ax=axes[1, 0])
axes[1, 0].axhline(1.0, color='gray', linestyle='--', alpha=0.7, label='Cuadrado (1.0)')
axes[1, 0].set_title('Distribucion de Relacion de Aspecto (W / H) por Clase', fontweight='bold')
axes[1, 0].set_xlabel('Clase')
axes[1, 0].set_ylabel('Aspect Ratio')
axes[1, 0].legend()

# 4. Boxplot de Resolucion Total (Megapixeles)
sns.boxplot(data=df_memes, x='categoria', y='megapixeles', palette=palette, ax=axes[1, 1])
axes[1, 1].set_title('Distribucion de Resolucion (Megapixeles) por Clase', fontweight='bold')
axes[1, 1].set_xlabel('Clase')
axes[1, 1].set_ylabel('Megapixeles (Area en Millones de px)')

plt.suptitle('Analisis de Diagnostico Dimensional: Deteccion de Sesgo de Caja por Categoria', 
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### Interpretacion de los Boxplots y Evaluacion del Riesgo

1. **Relacion de Aspecto (W / H)**:
   * Las medianas de las tres clases estan fuertemente alineadas cerca de 1.0 (cuadrado): `animado` (0.987), `animales` (0.956), `personas` (1.000).
   * Los rangos intercuartilicos (IQR) se solapan ampliamente.
   * **Conclusion**: La relacion de aspecto por si sola **no** le da a la red una regla facil para separar clases.

2. **Resolucion Total y Dimensiones (Ancho / Alto)**:
   * La categoria `animales` presenta una mediana de resolucion mayor (0.76 MP) en comparacion con `animado` (0.44 MP) y `personas` (0.52 MP).
   * La categoria `personas` contiene los valores extremos del dataset (tanto el meme de menor resolucion de 0.03 MP como el de mayor resolucion de 1.79 MP).
   * **Riesgo potencial**: Si una red se entrenara con imagenes sin normalizar o con redimensionamiento directo, podria correlacionar la nitidez o artefactos de alta resolucion con la clase `animales`.

3. **Como neutraliza esto nuestro preprocesamiento**:
   * Al aplicar **Letterboxing simetrico** y reescalar todas las imagenes a una matriz fija de $384 \times 384$ con interpolacion `LANCZOS`, todas las imagenes terminan con el mismo numero exacto de pixeles y rango de escala.
   * El relleno en gris neutro (128) asegura que las barras no aporten gradientes artificiales, forzando a la red a enfocarse exclusivamente en el contenido interno del meme.

## Fase 3: Comparativa Visual de Resoluciones (Alta vs Baja Calidad)

Mostramos visualmente como se ven las imagenes en diferentes extremos de resolucion dentro del dataset:
* El meme con la **resolucion mas baja** del dataset.
* Un meme con **resolucion tipica / media**.
* El meme con la **resolucion mas alta** del dataset.
* Su comparacion al ser adaptados al tamaño estandar de entrada de la CNN ($384 \times 384$).

In [ ]:
# Identificacion de casos de resolucion
min_res_row = df_memes.loc[df_memes['megapixeles'].idxmin()]
max_res_row = df_memes.loc[df_memes['megapixeles'].idxmax()]

mediana_mp = df_memes['megapixeles'].median()
med_res_row = df_memes.iloc[(df_memes['megapixeles'] - mediana_mp).abs().argsort()[:1]].iloc[0]

muestras_res = [
    ("Meme de Menor Resolucion (Baja Calidad)", min_res_row),
    ("Meme de Resolucion Media (Tipica)", med_res_row),
    ("Meme de Mayor Resolucion (Alta Calidad)", max_res_row)
]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, (titulo, fila) in zip(axes, muestras_res):
    with Image.open(fila['ruta_completa']) as img_raw:
        img_preview = img_raw.copy()
        img_preview.thumbnail((500, 500))
        ax.imshow(img_preview)
    ax.set_title(f"{titulo}\nClase: {fila['categoria']}\nDimensiones: {fila['ancho']}x{fila['alto']} px ({fila['megapixeles']} MP)\nAR: {fila['aspect_ratio']}", 
                 fontsize=11, fontweight='bold')
    ax.axis('off')

plt.suptitle('Comparativa Visual del Rango de Resoluciones en el Dataset Crudo', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Fase 4: Diagnostico de Problemas Criticos para la CNN

| # | Problema Potencial | Causa en Datos de Memes | Impacto en la CNN si no se atiende |
|---|---|---|---|
| **1** | **Formatos heterogéneos y extensiones engañosas** | Descargas en WebP, PNG, GIF o archivos PNG renombrados como JPG. | Decodificadores matriciales fallan o retornan valores nulos durante el entrenamiento. |
| **2** | **Inconsistencia de canales (RGBA, L, P, CMYK)** | Memes descargados como stickers contienen transparencia (canal Alfa, 4 canales) o blanco y negro (1 canal). En este dataset se detectaron 188 imagenes en modo RGBA. | Las capas convolucionales en TensorFlow esperan una forma fija `(Batch, H, W, 3)`. Un tensor de 4 canales rompe la red con error de dimensiones. |
| **3** | **Deformacion geometrica por Resize directo** | Relacion de aspecto variable desde 0.50 (vertical) hasta 1.78 (horizontal). | Forzar redimensionamiento plano a $384 \times 384$ aplasta caras y anatomia, deformando los patrones que buscan los filtros convolucionales. |
| **4** | **Perdida de informacion por Recorte Central (Center Crop)** | Aplicar recorte central para evitar deformacion elimina los margenes superior e inferior. | Se cortan textos superiores e inferiores (captions / remates) y partes de los sujetos. |
| **5** | **Orientacion oculta por metadatos EXIF** | Imagenes tomadas con camaras moviles guardan la orientacion en la etiqueta EXIF `0x0112`. | La red lee la matriz de lado o invertida si no se corrige antes de procesarla. |
| **6** | **Imagenes animadas (GIF / WebP)** | Memes con movimiento tienen multiples fotogramas. | Puede causar fallos de puntero de fotograma si no se selecciona explícitamente el primer frame. |

## Fase 5: Estudio Comparativo de Tecnicas de Estandarizacion

Para alimentar la CNN, evaluamos 4 metodos principales:
1. **Metodo A: Redimensionamiento Forzado (Stretch / Squish)**: Escala directamente sin respetar proporcion.
2. **Metodo B: Recorte Central (Center Crop)**: Escala la menor dimension y recorta los margenes.
3. **Metodo C: Letterboxing con Padding Simetrico**: Preserva 100% la proporcion rellenando con bordes simetricos.
   * Evaluamos: Relleno Gris Neutro (128, 128, 128), Relleno Negro (0, 0, 0) y Relleno con Reflejo.

### Sustento del Gris Neutro (128, 128, 128):
* Al aplicar la normalizacion estandar Z-Score $(X - \mu) / \sigma$ con medias de ImageNet $(\mu \approx 0.45)$, el valor 128 se convierte practicamente en cero $(Z \approx 0.0)$.
* El negro (0) se convierte en un valor anomalo de $-2.0\sigma$, provocando que los detectores de bordes de la capa Conv1 se activen fuertemente en el marco artificial.
* Es el estandar de la industria utilizado por arquitecturas como YOLOv5/v8/v11 (`color=(114, 114, 114)`).

In [ ]:
# Implementacion de las tecnicas de estandarizacion

def resize_forzado(img_pil, target_size=(384, 384)):
    return img_pil.resize(target_size, Image.Resampling.LANCZOS)

def center_crop(img_pil, target_size=(384, 384)):
    w, h = img_pil.size
    target_w, target_h = target_size
    escala = max(target_w / w, target_h / h)
    nw, nh = int(round(w * escala)), int(round(h * escala))
    
    img_redim = img_pil.resize((nw, nh), Image.Resampling.LANCZOS)
    left = (nw - target_w) // 2
    top = (nh - target_h) // 2
    return img_redim.crop((left, top, left + target_w, top + target_h))

def letterbox_padding(img_pil, target_size=(384, 384), pad_color=(128, 128, 128)):
    w, h = img_pil.size
    target_w, target_h = target_size
    escala = min(target_w / w, target_h / h)
    nw, nh = int(round(w * escala)), int(round(h * escala))
    
    img_redim = img_pil.resize((nw, nh), Image.Resampling.LANCZOS)
    
    canvas = Image.new('RGB', target_size, pad_color)
    paste_x = (target_w - nw) // 2
    paste_y = (target_h - nh) // 2
    canvas.paste(img_redim, (paste_x, paste_y))
    return canvas

def letterbox_reflejo(img_pil, target_size=(384, 384)):
    arr = np.array(img_pil)
    h, w = arr.shape[:2]
    target_w, target_h = target_size
    escala = min(target_w / w, target_h / h)
    nw, nh = int(round(w * escala)), int(round(h * escala))
    
    redim = cv2.resize(arr, (nw, nh), interpolation=cv2.INTER_AREA)
    pad_w = target_w - nw
    pad_h = target_h - nh
    top, bottom = pad_h // 2, pad_h - (pad_h // 2)
    left, right = pad_w // 2, pad_w - (pad_w // 2)
    
    reflejado = cv2.copyMakeBorder(redim, top, bottom, left, right, cv2.BORDER_REFLECT_101)
    return Image.fromarray(reflejado)

In [ ]:
# Comparativa visual sobre un caso vertical y uno horizontal
with Image.open(mas_vertical['ruta_completa']) as im_v:
    img_vert = im_v.convert('RGB')
    img_vert.thumbnail((400, 400))
with Image.open(mas_horizontal['ruta_completa']) as im_h:
    img_horiz = im_h.convert('RGB')
    img_horiz.thumbnail((400, 400))

ejemplos = [
    ("Meme Vertical (AR Minimo)", img_vert),
    ("Meme Horizontal (AR Maximo)", img_horiz)
]

tecnicas = [
    ("Original", lambda im: im),
    ("A. Resize Forzado", lambda im: resize_forzado(im)),
    ("B. Center Crop", lambda im: center_crop(im)),
    ("C. Letterbox (Gris 128)", lambda im: letterbox_padding(im, pad_color=(128, 128, 128))),
    ("C. Letterbox (Negro 0)", lambda im: letterbox_padding(im, pad_color=(0, 0, 0))),
    ("D. Letterbox Reflejo", lambda im: letterbox_reflejo(im))
]

fig, axes = plt.subplots(len(ejemplos), len(tecnicas), figsize=(18, 7))

for row_idx, (nombre_ej, img_raw) in enumerate(ejemplos):
    for col_idx, (nombre_tec, func) in enumerate(tecnicas):
        ax = axes[row_idx, col_idx]
        resultado = func(img_raw)
        ax.imshow(resultado)
        if row_idx == 0:
            ax.set_title(nombre_tec, fontweight='bold', fontsize=11)
        if col_idx == 0:
            ax.set_ylabel(nombre_ej, fontweight='bold', fontsize=11)
        ax.set_xticks([])
        ax.set_yticks([])

plt.suptitle("Comparativa de Tecnicas de Estandarizacion de Memes", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Fase 6: Pipeline de Preprocesamiento Robusto (RobustMemePreprocessor)

Diseño de la clase `RobustMemePreprocessor` modular en TensorFlow:
* Soporte universal de formatos (PNG, JPG, WEBP, GIF, BMP).
* Correccion automatica de orientacion EXIF.
* Extraccion del primer fotograma en imagenes animadas.
* Composicion alfa limpia para imagenes RGBA/transparencia (evitando fondos negros).
* Conversion garantizada a RGB de 3 canales.
* Letterboxing simetrico de alta calidad (LANCZOS).
* Salidas directas en PIL Image, NumPy array o tf.Tensor.

In [ ]:
class RobustMemePreprocessor:
    """
    Procesador universal de memes optimizado para TensorFlow / Keras.
    Maneja cualquier formato, canal de color, transparencia y relacion de aspecto.
    """
    def __init__(self, target_size=(384, 384), pad_color=(128, 128, 128)):
        self.target_size = target_size
        self.pad_color = pad_color
        
    def process_pil(self, image_input) -> Image.Image:
        if isinstance(image_input, (str, Path)):
            img = Image.open(image_input)
        elif isinstance(image_input, Image.Image):
            img = image_input.copy()
        else:
            raise TypeError(f"Entrada no soportada: {type(image_input)}")
            
        if getattr(img, 'is_animated', False):
            img.seek(0)
            
        img = ImageOps.exif_transpose(img)
        
        if img.mode in ('RGBA', 'LA'):
            background = Image.new('RGB', img.size, self.pad_color)
            alpha_mask = img.split()[-1]
            background.paste(img.convert('RGB'), mask=alpha_mask)
            img = background
        elif img.mode == 'P':
            img = img.convert('RGBA')
            background = Image.new('RGB', img.size, self.pad_color)
            background.paste(img.convert('RGB'), mask=img.split()[-1])
            img = background
        elif img.mode != 'RGB':
            img = img.convert('RGB')
            
        w, h = img.size
        target_w, target_h = self.target_size
        escala = min(target_w / w, target_h / h)
        nw, nh = max(1, int(round(w * escala))), max(1, int(round(h * escala)))
        
        img_redim = img.resize((nw, nh), Image.Resampling.LANCZOS)
        
        canvas = Image.new('RGB', self.target_size, self.pad_color)
        paste_x = (target_w - nw) // 2
        paste_y = (target_h - nh) // 2
        canvas.paste(img_redim, (paste_x, paste_y))
        
        return canvas

    def process_numpy(self, image_input) -> np.ndarray:
        img_pil = self.process_pil(image_input)
        return np.array(img_pil, dtype=np.float32)

    def process_tensor(self, image_input) -> tf.Tensor:
        arr = self.process_numpy(image_input)
        return tf.convert_to_tensor(arr, dtype=tf.float32)

preprocesador = RobustMemePreprocessor(target_size=(384, 384), pad_color=(128, 128, 128))
print("RobustMemePreprocessor inicializado y listo para TensorFlow.")

## Fase 7: Estandarizacion de Nombres de Archivo por Clase

### El Problema de la Nomenclatura Dispar en Memes:
Al inspeccionar los archivos crudos encontramos inconsistencias notables entre las carpetas:
* `animales`: Contiene nombres con numeros simples (`1.png`, `2.png`, etc.) combinados con `animal_01.png`.
* `personas`: Contiene nombres como `meme.png`, `meme1.png`, `meme10.png`, etc.
* `animado`: Contiene nombres con prefijo `animado_01.png`.

Esto genera problemas en ordenamiento alfabetico en sistemas operativos (por ejemplo, `meme10.png` se ordena antes de `meme2.png` si no tienen ceros a la izquierda), dificultando la reproducibilidad de particiones de entrenamiento y validacion.

### Solucion Estandarizada:
Definimos un esquema de nomenclatura canonico y con ceros a la izquierda:
$$\text{nombre\_estandarizado} = \text{categoria} \_ \{idx:03d\} . \text{jpg}$$
Por ejemplo:
* `animado_001.jpg` hasta `animado_250.jpg`
* `animales_001.jpg` hasta `animales_250.jpg`
* `personas_001.jpg` hasta `personas_250.jpg`

Ademas, creamos una tabla de mapeo y trazabilidad (`mapeo_nombres.csv`) para poder rastrear con exactitud que archivo original corresponde a cada imagen procesada.

In [ ]:
# Generacion de la tabla de mapeo de nombres estandarizados
registros_mapeo = []

for categoria in CATEGORIAS:
    dir_cat = BASE_DIR / categoria
    archivos = sorted([f for f in dir_cat.iterdir() if f.is_file()])
    
    for idx, f in enumerate(archivos, start=1):
        nombre_estandar = f"{categoria}_{idx:03d}.jpg"
        registros_mapeo.append({
            'categoria': categoria,
            'indice': idx,
            'nombre_original': f.name,
            'nombre_estandarizado': nombre_estandar,
            'ruta_original': str(f)
        })

df_mapeo = pd.DataFrame(registros_mapeo)

print(f"Total de nombres estandarizados preparados: {len(df_mapeo)}")
print("\nMuestra del mapeo canonico (primeros 5 por clase):")
for cat in CATEGORIAS:
    print(f"\n--- Mapeo para clase '{cat}' ---")
    sub = df_mapeo[df_mapeo['categoria'] == cat].head(5)[['nombre_original', 'nombre_estandarizado']]
    print(sub.to_string(index=False))

## Fase 8: Generacion del Dataset Fisico Estandarizado (dataset_preprocesado/)

Exportamos todo el dataset estandarizado al directorio `dataset_preprocesado/` aplicando:
1. Resolucion exacta de **384 x 384 px**.
2. **3 canales de color (RGB)** con composicion limpia de transparencias.
3. **Nombres de archivo completamente estandarizados** (`animado_001.jpg`, `animales_001.jpg`, `personas_001.jpg`).
4. Archivo de registro `mapeo_nombres.csv` guardado en la raiz de `dataset_preprocesado/` para trazabilidad completa.

Estructura resultante:
```
dataset_preprocesado/
├── mapeo_nombres.csv
├── animado/    (animado_001.jpg ... animado_250.jpg)
├── animales/   (animales_001.jpg ... animales_250.jpg)
└── personas/   (personas_001.jpg ... personas_250.jpg)
```

In [ ]:
# Generacion del dataset estandarizado en disco con nombres canonicos
DIR_SALIDA = Path('dataset_preprocesado')
DIR_SALIDA.mkdir(exist_ok=True)

# Verificar si el dataset ya fue generado previamente con resolucion exacta 384x384
TARGET_SIZE = (384, 384)
primer_img = next(DIR_SALIDA.glob('*/*.jpg'), None)
resolucion_correcta = False
if primer_img:
    with Image.open(primer_img) as im:
        resolucion_correcta = (im.size == TARGET_SIZE)

conteo_existente = len(list(DIR_SALIDA.glob('*/*.jpg')))
if conteo_existente >= 750 and resolucion_correcta:
    print(f"El dataset estandarizado ({TARGET_SIZE[0]}x{TARGET_SIZE[1]}) ya existe en '{DIR_SALIDA.resolve()}' con {conteo_existente} imagenes.")
    print("Omitiendo reprocesamiento para evitar redundancia y ahorrar tiempo.")
else:
    print(f"Generando dataset estandarizado en alta resolucion {TARGET_SIZE}...")
    # Guardar tabla de mapeo CSV para trazabilidad
    ruta_csv_mapeo = DIR_SALIDA / "mapeo_nombres.csv"
    df_mapeo.to_csv(ruta_csv_mapeo, index=False, encoding='utf-8')
    print(f"Archivo de trazabilidad guardado en: {ruta_csv_mapeo}")

    estadisticas_procesamiento = defaultdict(int)
    errores_exportacion = []

    for _, fila in df_mapeo.iterrows():
        cat = fila['categoria']
        ruta_orig = Path(fila['ruta_original'])
        nombre_nuevo = fila['nombre_estandarizado']
        
        dir_salida_cat = DIR_SALIDA / cat
        dir_salida_cat.mkdir(exist_ok=True)
        
        try:
            img_estandarizada = preprocesador.process_pil(ruta_orig)
            ruta_destino = dir_salida_cat / nombre_nuevo
            img_estandarizada.save(ruta_destino, format='JPEG', quality=95)
            estadisticas_procesamiento[cat] += 1
        except Exception as e:
            errores_exportacion.append((str(ruta_orig), str(e)))

    print("\nResumen de Generacion del Dataset Estandarizado:")
    for cat, count in estadisticas_procesamiento.items():
        print(f"  * Categoria '{cat}': {count} imagenes procesadas a (384, 384, 3) con nombres {cat}_001.jpg a {cat}_{count:03d}.jpg")

    print(f"\nTotal procesado con exito: {sum(estadisticas_procesamiento.values())}")
    print(f"Total con errores: {len(errores_exportacion)}")

## Fase 9: Data Augmentation 100% en CPU con OpenCV (Anti-Esquinas Negras y GPU-Friendly)

### Fundamento de Rendimiento en GPUs AMD (GPU)
En TensorFlow con GPU, capas de red como `layers.RandomContrast` o generadores aleatorios internos no poseen kernels nativos DirectX 12. Esto obliga a TensorFlow a activar un bucle de reserva secuencial en CPU (`while_loop for converting AdjustContrastv2`), transfiriendo continuamente los datos por el bus PCIe y disparando los tiempos de paso de **~70 ms a más de 800 ms**.

### Solucion Optimizada: Aumento de Datos en CPU con OpenCV
1. **Transformaciones en C++ (< 0.25 ms/imagen)**: Rotación, escala y ajustes de contraste/brillo se ejecutan en CPU antes de transferir el lote a la GPU.
2. **Solucion al Problema de Esquinas Negras**: En `cv2.warpAffine`, fijamos `borderMode=cv2.BORDER_CONSTANT` y `borderValue=(128, 128, 128)` (gris neutro). De esta forma, cualquier zona descubierta se fusiona imperceptiblemente con el marco Letterbox y, tras la normalización al rango $[-1, 1]$, equivale exactamente a **0.0**.
3. **Grafo Limpio en GPU**: La red neuronal recibe lotes ya terminados y opera al 100% de la velocidad de los Compute Units de la GPU.

In [ ]:
# Definicion de funciones de Data Augmentation en CPU con OpenCV
def augment_meme_cpu(img_rgb):
    """
    Aplica Data Augmentation en CPU con OpenCV optimizado para GPU.
    Incluye traslacion leve (+/-6%) y preserva relleno gris neutro (128, 128, 128).
    """
    img = img_rgb.copy()
    h, w = img.shape[:2]
    
    # 1. Volteo horizontal aleatorio (50% de probabilidad)
    if np.random.rand() > 0.5:
        img = cv2.flip(img, 1)
        
    # 2. Rotacion (+/- 6 grados), zoom (0.94 a 1.06) y traslacion (+/- 6%)
    angle = np.random.uniform(-6.0, 6.0)
    scale = np.random.uniform(0.94, 1.06)
    tx = np.random.uniform(-0.06 * w, 0.06 * w)
    ty = np.random.uniform(-0.06 * h, 0.06 * h)
    
    M = cv2.getRotationMatrix2D((w / 2.0, h / 2.0), angle, scale)
    M[0, 2] += tx
    M[1, 2] += ty
    
    img = cv2.warpAffine(
        img, M, (w, h),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(128, 128, 128) # Relleno gris neutro anti-esquinas negras
    )
    
    # 3. Variaciones fotometricas de contraste y brillo
    alpha = np.random.uniform(0.82, 1.18)
    beta = np.random.uniform(-12, 12)
    img = cv2.convertScaleAbs(img, alpha=alpha, beta=beta)
    
    # 4. Micro-Cutout estocastico reforzado (1 o 2 parches con relleno gris neutro 128)
    if np.random.rand() > 0.35:
        num_patches = 1 if np.random.rand() > 0.4 else 2
        for _ in range(num_patches):
            ch = np.random.randint(int(0.08 * h), int(0.18 * h))
            cw = np.random.randint(int(0.08 * w), int(0.18 * w))
            cy = np.random.randint(10, max(11, h - ch - 10))
            cx = np.random.randint(10, max(11, w - cw - 10))
            img[cy:cy+ch, cx:cx+cw] = (128, 128, 128)
            
    return img

print("Pipeline de Data Augmentation 100% en CPU (OpenCV + Relleno Gris Neutro 128) configurado.")

In [ ]:
# Demostracion visual de Data Augmentation en CPU con muestras reales
DIR_DATASET = Path('dataset_preprocesado')
muestra_rutas = list(DIR_DATASET.glob("*/*.jpg"))[:4]

fig, axes = plt.subplots(len(muestra_rutas), 4, figsize=(15, 3.2 * len(muestra_rutas)))

for row_idx, ruta in enumerate(muestra_rutas):
    img_orig = cv2.cvtColor(cv2.imread(str(ruta)), cv2.COLOR_BGR2RGB)
    
    axes[row_idx, 0].imshow(img_orig)
    axes[row_idx, 0].set_title(f"Original ({ruta.parent.name})", fontweight='bold')
    axes[row_idx, 0].axis('off')
    
    for col_idx in range(1, 4):
        img_aug = augment_meme_cpu(img_orig)
        axes[row_idx, col_idx].imshow(img_aug)
        axes[row_idx, col_idx].set_title(f"Aumento CPU #{col_idx}", fontsize=10)
        axes[row_idx, col_idx].axis('off')

plt.suptitle("Demostracion de Data Augmentation en CPU con OpenCV (Preservando Gris Neutro 128)",
             fontsize=14, fontweight='bold', y=0.99)
plt.tight_layout()
plt.show()

## Fase 10: Precarga en Memoria RAM (Latencia 0 ms) y Generador de Lotes (Batch Size = 32)

### Fundamentos de Arquitectura GPU:
1. **Cache en Memoria RAM (0 ms)**: Las 750 imagenes estandarizadas ($224 \times 224 \times 3$ uint8) pesan apenas **~332 MB en RAM**. Al precargar el dataset completo en memoria al inicio, eliminamos el costo de 5 a 10 ms por lectura de archivo en disco en cada epoca.
2. **Particion Estratificada 80/20**: 600 imagenes de entrenamiento (200 por clase) y 150 de validacion (50 por clase).
3. **Batch Size = 32**: Fija la saturacion optima de Compute Units (~60%) en la GPU GPU Acelerada (NVIDIA T4 / Colab) y previene desbordamientos de asignacion virtual del controlador GPU (error `Ioctl -22`).
4. **`tf.keras.utils.Sequence`**: Generador multihilo en CPU que entrega tensores `float32` limpios listos para el grafo GPU.

In [ ]:
# 1. Precarga completa en Memoria RAM
print("Iniciando precarga del dataset en memoria RAM...")
clases = sorted([d.name for d in DIR_DATASET.iterdir() if d.is_dir()])
class_names = clases
class_to_idx = {cls_name: i for i, cls_name in enumerate(class_names)}

X_list = []
y_list = []

for cls_name in class_names:
    cls_dir = DIR_DATASET / cls_name
    rutas_img = sorted(list(cls_dir.glob("*.jpg")) + list(cls_dir.glob("*.png")))
    for r in rutas_img:
        img_bgr = cv2.imread(str(r))
        if img_bgr is not None:
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            X_list.append(img_rgb)
            y_list.append(class_to_idx[cls_name])

X_all = np.array(X_list, dtype=np.uint8)
y_indices = np.array(y_list, dtype=np.int32)
y_all = tf.keras.utils.to_categorical(y_indices, num_classes=len(class_names))

print(f"Dataset precargado exitosamente en RAM:")
print(f"  * Total de imagenes: {len(X_all)} | Forma de tensores: {X_all.shape}")
print(f"  * Memoria RAM utilizada: {X_all.nbytes / (1024 * 1024):.2f} MB")
print(f"  * Clases detectadas: {class_names}")

# 2. Particion estratificada 80/20
SEED = 42
BATCH_SIZE = 32

X_train, X_val, y_train, y_val = train_test_split(
    X_all, y_all, test_size=0.20, random_state=SEED, stratify=y_indices
)

print(f"\nParticion estratificada completada:")
print(f"  * Entrenamiento (80%): {len(X_train)} imagenes ({int(np.ceil(len(X_train)/BATCH_SIZE))} lotes de {BATCH_SIZE})")
print(f"  * Validacion (20%):   {len(X_val)} imagenes ({int(np.ceil(len(X_val)/BATCH_SIZE))} lotes de {BATCH_SIZE})")

# 3. Generador de datos en CPU con tf.keras.utils.Sequence
class MemeDataGenerator(tf.keras.utils.Sequence):
    """Generador en CPU con precarga en RAM y aumento paralelo."""
    def __init__(self, images, labels, batch_size=32, shuffle=True, augment=False):
        self.images = images
        self.labels = labels
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.augment = augment
        self.indices = np.arange(len(self.images))
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __len__(self):
        return int(np.ceil(len(self.images) / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size : (idx + 1) * self.batch_size]
        batch_x = self.images[batch_idx]
        batch_y = self.labels[batch_idx]

        if self.augment:
            batch_x_aug = [augment_meme_cpu(img) for img in batch_x]
            batch_x = np.array(batch_x_aug, dtype=np.uint8)
        else:
            batch_x = batch_x.astype(np.uint8)

        return batch_x, batch_y

train_generator = MemeDataGenerator(X_train, y_train, batch_size=BATCH_SIZE, shuffle=True, augment=True)
val_generator = MemeDataGenerator(X_val, y_val, batch_size=BATCH_SIZE, shuffle=False, augment=False)
print("Generadores MemeDataGenerator (CPU + Cache RAM + Batch 32) configurados.")

## Fase 11: Construccion de la Arquitectura CNN Nativa en GPU (Grafo Limpio en GPU)

### Principios de Diseño para GPU / DirectX 12:
1. **Grafo Limpio en GPU**: La red neuronal no incluye capas aleatorias internas de Data Augmentation (`RandomContrast`, `RandomRotation`), garantizando que cada operacion se ejecute 100% nativa en los Compute Units de la GPU.
2. **Normalizacion Simetrica $[-1, 1]$**: `Rescaling(scale=1./127.5, offset=-1.0)` convierte el fondo gris neutro ($128$) exactamente en **$0.0$**, neutralizando las activaciones en el marco letterbox.
3. **4 Bloques Convolucionales con Batch Normalization (momentum=0.9)**: Normaliza activaciones intermedias con convergencia suave en GPU.
4. **Global Average Pooling 2D**: Reemplaza Flatten y reduce el total de parametros a $\approx 107$ mil, previniendo el sobreajuste.
5. **Monitoreo Continuo de Metricas**: Monitoreo en vivo de **Accuracy**, **Precision**, **Recall** y **F1-Score**.

In [ ]:
# Metrica personalizada de F1-Score continuo para Keras y definicion de la arquitectura
import tensorflow as tf
from tensorflow.keras import layers, models
import gc

class F1ScoreMetric(tf.keras.metrics.Metric):
    """Metrica serializable para calcular F1-Score continuo en tiempo de entrenamiento."""
    def __init__(self, name='f1_score', **kwargs):
        super().__init__(name=name, **kwargs)
        self.precision_op = tf.keras.metrics.Precision()
        self.recall_op = tf.keras.metrics.Recall()

    def update_state(self, y_true, y_pred, sample_weight=None):
        self.precision_op.update_state(y_true, y_pred, sample_weight)
        self.recall_op.update_state(y_true, y_pred, sample_weight)

    def result(self):
        p = self.precision_op.result()
        r = self.recall_op.result()
        return tf.math.divide_no_nan(2.0 * p * r, p + r)

    def reset_state(self):
        self.precision_op.reset_state()
        self.recall_op.reset_state()

def build_custom_cnn(input_shape=(384, 384, 3), num_classes=3):
    l2_reg = tf.keras.regularizers.l2(1e-4)
    inputs = layers.Input(shape=input_shape, dtype='uint8', name="input_imagen")
    
    # Normalizacion simetrica al rango [-1, 1] (Gris 128 -> 0.0)
    x = layers.Rescaling(scale=1./127.5, offset=-1.0, name="normalizacion_gris_cero")(inputs)
    
    # Bloque 1: Filtros 5x5 para trazos base
    x = layers.Conv2D(16, (5, 5), padding='same', kernel_regularizer=l2_reg, use_bias=False, name="conv_bloque1")(x)
    x = layers.BatchNormalization(momentum=0.9, name="bn_bloque1")(x)
    x = layers.ReLU(name="relu_bloque1")(x)
    x = layers.MaxPooling2D((2, 2), name="pool_bloque1")(x)
    
    # Bloque 2: 32 filtros 3x3
    x = layers.Conv2D(32, (3, 3), padding='same', kernel_regularizer=l2_reg, use_bias=False, name="conv_bloque2")(x)
    x = layers.BatchNormalization(momentum=0.9, name="bn_bloque2")(x)
    x = layers.ReLU(name="relu_bloque2")(x)
    x = layers.MaxPooling2D((2, 2), name="pool_bloque2")(x)
    
    # Bloque 3: 64 filtros 3x3
    x = layers.Conv2D(64, (3, 3), padding='same', kernel_regularizer=l2_reg, use_bias=False, name="conv_bloque3")(x)
    x = layers.BatchNormalization(momentum=0.9, name="bn_bloque3")(x)
    x = layers.ReLU(name="relu_bloque3")(x)
    x = layers.MaxPooling2D((2, 2), name="pool_bloque3")(x)
    
    # Bloque 4: 128 filtros 3x3
    x = layers.Conv2D(128, (3, 3), padding='same', kernel_regularizer=l2_reg, use_bias=False, name="conv_bloque4")(x)
    x = layers.BatchNormalization(momentum=0.9, name="bn_bloque4")(x)
    x = layers.ReLU(name="relu_bloque4")(x)
    x = layers.MaxPooling2D((2, 2), name="pool_bloque4")(x)
    
    # Cuello de botella
    x = layers.GlobalAveragePooling2D(name="global_avg_pool")(x)
    
    # Clasificador denso con L2 y Dropout
    x = layers.Dense(64, activation='relu', kernel_regularizer=l2_reg, use_bias=False, name="dense_oculta")(x)
    x = layers.BatchNormalization(momentum=0.9, name="bn_densa")(x)
    x = layers.Dropout(0.5, name="dropout_regularizacion")(x)
    outputs = layers.Dense(num_classes, activation='softmax', name="salida_softmax")(x)
    
    model = tf.keras.Model(inputs=inputs, outputs=outputs, name="CNN_Memes_Baseline")
    return model

tf.keras.backend.clear_session()
gc.collect()

num_clases_detectadas = len(class_names) if 'class_names' in globals() else 3
modelo_cnn = build_custom_cnn(input_shape=(384, 384, 3), num_classes=num_clases_detectadas)
print("Resumen de la arquitectura CNN construida (Grafo Limpio GPU):")
modelo_cnn.summary()

## Fase 12: Compilacion y Entrenamiento del Modelo Baseline

Monitoreo en vivo de **Loss**, **Accuracy**, **Precision**, **Recall** y **F1-Score**, con regularizacion y respaldos automaticos.

In [ ]:
# Compilacion y entrenamiento optimizado
import os
import json
from pathlib import Path
import tensorflow as tf
from tensorflow.keras import callbacks

ruta_baseline = Path("modelo_memes_baseline.keras")
ruta_historia_baseline = Path("historia_baseline.json")

EPOCHS = 20

TARGET_SHAPE = (None, 384, 384, 3)
baseline_cargado = False

if ruta_baseline.exists():
    try:
        candidato = tf.keras.models.load_model(ruta_baseline, custom_objects={'F1ScoreMetric': F1ScoreMetric, 'AdamW': tf.keras.optimizers.AdamW}, compile=False)
        if candidato.input_shape == TARGET_SHAPE:
            modelo_cnn = candidato
            baseline_cargado = True
            print(f"Cargando modelo Baseline a 384x384 desde: {ruta_baseline.resolve()}")
            if ruta_historia_baseline.exists():
                with open(ruta_historia_baseline, "r", encoding="utf-8") as f:
                    historia_dict = json.load(f)
                class HistoryWrapper:
                    def __init__(self, d):
                        self.history = d
                historia = HistoryWrapper(historia_dict)
            else:
                historia = None
            print("Modelo Baseline cargado con exito. Omitiendo reentrenamiento para ahorrar recursos.")
        else:
            print(f"Aviso: Archivo '{ruta_baseline.name}' tiene forma {candidato.input_shape} != {TARGET_SHAPE}.")
            print("Entrenando nuevo modelo Baseline adaptado a 384x384...")
    except Exception as e:
        print(f"Aviso al cargar modelo previo: {e}. Se procedera a entrenar.")

if not baseline_cargado:
    modelo_cnn = build_custom_cnn(input_shape=(384, 384, 3), num_classes=num_clases_detectadas)
    modelo_cnn.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
        metrics=[
            'accuracy',
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall'),
            F1ScoreMetric(name='f1_score')
        ]
    )

    callbacks_list = [
        callbacks.EarlyStopping(
            monitor='val_loss',
            patience=6,
            restore_best_weights=True,
            verbose=1
        ),
        callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=3,
            min_lr=1e-5,
            verbose=1
        )
    ]

    print("Iniciando entrenamiento del modelo Baseline acelerado en GPU GPU...")
    historia = modelo_cnn.fit(
        train_generator,
        validation_data=val_generator,
        epochs=EPOCHS,
        callbacks=callbacks_list,
        verbose=1
    )
    print("Entrenamiento finalizado.")
    
    # Guardado persistente del modelo y su historial
    modelo_cnn.save(ruta_baseline)
    print(f"Modelo Baseline guardado exitosamente en: {ruta_baseline.resolve()}")
    with open(ruta_historia_baseline, "w", encoding="utf-8") as f:
        json.dump({k: [float(v) for v in vals] for k, vals in historia.history.items()}, f)
    print(f"Historial del Baseline guardado en: {ruta_historia_baseline.resolve()}")

## Fase 13: Evaluacion del Modelo y Curvas de Aprendizaje Multimetrica

Visualizamos la evolucion simultanea de **Loss**, **Accuracy**, **Precision/Recall** y **F1-Score**.

In [ ]:
# Graficas de curvas de aprendizaje completas (Accuracy, Loss, Precision, Recall, F1)
import json
from pathlib import Path
import matplotlib.pyplot as plt

if 'historia' not in globals() or historia is None:
    ruta_h = Path("historia_baseline.json")
    if ruta_h.exists():
        with open(ruta_h, "r", encoding="utf-8") as f:
            hist_data = json.load(f)
    else:
        hist_data = {}
else:
    hist_data = historia.history

acc = hist_data.get('accuracy', [])
val_acc = hist_data.get('val_accuracy', [])
loss = hist_data.get('loss', [])
val_loss = hist_data.get('val_loss', [])
val_prec = hist_data.get('val_precision', [])
val_rec = hist_data.get('val_recall', [])
f1_tr = hist_data.get('f1_score', [])
val_f1 = hist_data.get('val_f1_score', [])
epochs_range = range(1, len(acc) + 1)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Accuracy
axes[0, 0].plot(epochs_range, acc, 'o-', color='#4C72B0', label='Entrenamiento')
axes[0, 0].plot(epochs_range, val_acc, 's--', color='#55A868', label='Validacion')
axes[0, 0].set_title('Precision (Accuracy)', fontweight='bold')
axes[0, 0].set_xlabel('Epoca')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].legend(loc='lower right')
axes[0, 0].set_ylim([0, 1.05])

# 2. Loss
axes[0, 1].plot(epochs_range, loss, 'o-', color='#C44E52', label='Entrenamiento')
axes[0, 1].plot(epochs_range, val_loss, 's--', color='#8172B2', label='Validacion')
axes[0, 1].set_title('Perdida (Cross-Entropy Loss)', fontweight='bold')
axes[0, 1].set_xlabel('Epoca')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend(loc='upper right')

# 3. Precision vs Recall en Validacion
if val_prec and val_rec:
    axes[1, 0].plot(epochs_range, val_prec, '^-', color='#E67E22', label='Val Precision')
    axes[1, 0].plot(epochs_range, val_rec, 'v--', color='#2980B9', label='Val Recall')
    axes[1, 0].set_title('Precision vs Recall (Validacion)', fontweight='bold')
    axes[1, 0].set_xlabel('Epoca')
    axes[1, 0].set_ylabel('Metrica')
    axes[1, 0].legend(loc='lower right')
    axes[1, 0].set_ylim([0, 1.05])

# 4. F1-Score
if val_f1:
    if f1_tr:
        axes[1, 1].plot(epochs_range, f1_tr, 'o-', color='#34495E', label='Train F1')
    axes[1, 1].plot(epochs_range, val_f1, 's--', color='#16A085', label='Val F1')
    axes[1, 1].set_title('F1-Score Continuo', fontweight='bold')
    axes[1, 1].set_xlabel('Epoca')
    axes[1, 1].set_ylabel('F1-Score')
    axes[1, 1].legend(loc='lower right')
    axes[1, 1].set_ylim([0, 1.05])

plt.suptitle('Curvas de Aprendizaje Multimetrica (GPU / GPU Acelerado)', fontsize=15, fontweight='bold', y=0.99)
plt.tight_layout()
plt.show()

## Fase 14: Matriz de Confusion y Metricas Oficiales (Accuracy, Precision, Recall, F1)

Evaluacion integral sobre el conjunto de validacion completo (150 imagenes en RAM).

In [ ]:
# Evaluacion de validacion completa: Accuracy, Precision, Recall y F1-Score
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score

# Predicciones con el generador de validacion
y_pred_probs = modelo_cnn.predict(val_generator, verbose=0)
y_predichas = np.argmax(y_pred_probs, axis=1)
y_reales = np.argmax(y_val, axis=1)
labels_idx = list(range(len(class_names)))

acc_baseline = float((y_reales == y_predichas).mean() * 100)
prec_macro_baseline = float(precision_score(y_reales, y_predichas, average='macro', zero_division=0) * 100)
rec_macro_baseline = float(recall_score(y_reales, y_predichas, average='macro', zero_division=0) * 100)
f1_macro_baseline = float(f1_score(y_reales, y_predichas, average='macro', zero_division=0) * 100)
f1_weighted_baseline = float(f1_score(y_reales, y_predichas, average='weighted', zero_division=0) * 100)

print("="*65)
print("DESEMPENO OFICIAL MODELO BASELINE (PARTE I):")
print(f"  * Accuracy:           {acc_baseline:.2f}%")
print(f"  * Precision (Macro):  {prec_macro_baseline:.2f}%")
print(f"  * Recall (Macro):     {rec_macro_baseline:.2f}%")
print(f"  * F1-Score Macro:     {f1_macro_baseline:.2f}%")
print(f"  * F1-Score Ponderado: {f1_weighted_baseline:.2f}%")
print("="*65)

# Reporte de clasificacion detallado
print("\nReporte de Clasificacion Detallado en Validacion (80/20):")
print(classification_report(y_reales, y_predichas, labels=labels_idx, target_names=class_names, zero_division=0))

# Matriz de confusion
cm = confusion_matrix(y_reales, y_predichas, labels=labels_idx)

plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names, cbar=False)
plt.title(f'Matriz de Confusion Baseline - F1 Macro: {f1_macro_baseline:.2f}%', fontweight='bold')
plt.xlabel('Clase Predicha por la CNN')
plt.ylabel('Clase Real (Ground Truth)')
plt.tight_layout()
plt.show()

## Fase 15: Inferencia Visual en Muestras de Validacion

In [ ]:
# Inferencia visual en un lote de validacion
imagenes_val, etiquetas_val = val_generator[0]
predicciones_prob = modelo_cnn.predict(imagenes_val, verbose=0)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i, ax in enumerate(axes.flat):
    if i < len(imagenes_val):
        img_np = np.clip(imagenes_val[i], 0, 255).astype(np.uint8)
        clase_real_idx = np.argmax(etiquetas_val[i])
        clase_pred_idx = np.argmax(predicciones_prob[i])
        
        clase_real = class_names[clase_real_idx]
        clase_pred = class_names[clase_pred_idx]
        confianza = predicciones_prob[i][clase_pred_idx] * 100
        
        es_correcta = (clase_real_idx == clase_pred_idx)
        color_titulo = '#2ca02c' if es_correcta else '#d62728'
        
        ax.imshow(img_np)
        ax.set_title(f"Pred: {clase_pred} ({confianza:.1f}%)\nReal: {clase_real}", 
                     color=color_titulo, fontweight='bold', fontsize=10)
    ax.axis('off')

plt.suptitle('Predicciones del Primer Modelo CNN sobre Imagenes de Validacion (GPU)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Conclusiones del Pipeline Completo de Preprocesamiento y Modelado

1. **Preprocesamiento y Estandarizacion Integral**:
   * Las 750 imagenes originales fueron convertidas a resolucion fija $384 \times 384$ con 3 canales RGB.
   * La composicion alfa limpia resolvio las 188 imagenes con canal de transparencia sin artefactos oscuros.
   * El relleno en gris neutro (128) asegura que el marco represente matematicamente el valor neutral ($0.0$) tras la normalizacion.
   * Se estandarizaron los nombres de archivo (`clase_001.jpg` a `clase_250.jpg`) con archivo de trazabilidad `mapeo_nombres.csv`.

2. **Pipeline de Modelado CNN**:
   * Particion estratificada 80/20 ($600$ imagenes para entrenamiento y $150$ para validacion).
   * Data Augmentation en Keras con rotacion y zoom usando `fill_value=128.0` eliminando las esquinas negras.
   * Arquitectura de 4 bloques convolucionales con Batch Normalization y Global Average Pooling con solo 107 mil parametros, previniendo el sobreajuste.
   * Callbacks de EarlyStopping y ReduceLROnPlateau para convergencia automatica.

# ==============================================================================
# PARTE II: Busqueda Sistematica con Conexiones de Atajo Residual y Resolucion 384x384 (GPU)
# ==============================================================================

### Diagnostico y Cambios Aplicados en esta Suite Avanzada:
1. **Resolucion de Alta Fidelidad ($384 	imes 384$ px, $+194\%$ de informacion)**:
   * Casi el triple de pixeles que el baseline de 224 px, resolviendo por completo la degradacion de textos finos y detalles faciales por letterboxing.
2. **Topologia Convolucional con Conexiones de Atajo Residual (`layers.Add`)**:
   $$\text{Salida} = \text{ReLU}(\text{Atajo}(x) + \text{Conv2D}(\text{ReLU}(\text{Conv2D}(x))))$$
   * Si el numero de canales aumenta, el atajo se proyecta limpiamente con `Conv2D(1x1)` y `BatchNormalization`.
   * Las conexiones de salto permiten el flujo irrestricto del gradiente hacia las primeras capas, rompiendo la barrera de underfitting observada en redes secuenciales.
3. **Eficiencia Estructural sin Sesgo (`use_bias=False` antes de BN)**:
   * Todas las capas convolucionales y densas previas a `BatchNormalization` operan con `use_bias=False`, ahorrando parametros redundantes y estabilizando el gradiente.
4. **Transferencia Optimizada `uint8` y Normalizacion en GPU (`layers.Rescaling`)**:
   * El generador transfiere matrices compactas `uint8` (1 byte/px), reduciendo en $4\times$ la saturacion del bus PCIe.
   * La primera capa del grafo escala a $[-1.0, 1.0]$, convirtiendo el gris de relleno (128) exactamente a $0.0$.
5. **Schedule Adaptativo con Calentamiento Lineal y Decaimiento por Coseno (`Cosine Decay`)**:
   * Warmup lineal en las 3 primeras epocas para asentar las estadisticas de `BatchNormalization`.
   * Decaimiento cosenoidal suave hasta la epoca 30, guiando los pesos hacia minimos planos de maxima generalizacion.
6. **Optimizador `AdamW` con Decoupled Weight Decay y Label Smoothing Calibrado (`0.03`)**:
   * Desacopla el decaimiento de pesos de la magnitud del gradiente.

In [ ]:
# Infraestructura modular integrada: Bloques con Atajos Residuales y Warmup/Cosine Schedule
import os
import gc
import sys
import time
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score

DIR_PROYECTO = Path.cwd()
RUTA_CACHE_TUNING = DIR_PROYECTO / "tuning_results.json"
DIR_CHECKPOINTS = DIR_PROYECTO / "checkpoints"
DIR_CHECKPOINTS.mkdir(exist_ok=True)

historial_experimentos = []

# Cargar experimentos previos si ya existen en cache
if RUTA_CACHE_TUNING.exists():
    try:
        with open(RUTA_CACHE_TUNING, "r", encoding="utf-8") as f:
            cache_cargada = json.load(f)
            historial_experimentos = list(cache_cargada.values())
        print(f"Se cargaron {len(historial_experimentos)} experimentos previos desde cache ({RUTA_CACHE_TUNING.name}).")
    except Exception as e:
        print(f"Advertencia al leer cache: {e}")

# Metrica F1-Score Macro en tiempo real para Keras
class F1ScoreMetric(tf.keras.metrics.Metric):
    def __init__(self, name='f1_score', **kwargs):
        super().__init__(name=name, **kwargs)
        self.precision = tf.keras.metrics.Precision()
        self.recall = tf.keras.metrics.Recall()

    def update_state(self, y_true, y_pred, sample_weight=None):
        self.precision.update_state(y_true, y_pred, sample_weight)
        self.recall.update_state(y_true, y_pred, sample_weight)

    def result(self):
        p = self.precision.result()
        r = self.recall.result()
        return tf.math.divide_no_nan(2.0 * p * r, p + r)

    def reset_state(self):
        self.precision.reset_state()
        self.recall.reset_state()


def schedule_cosine_warmup(epoch, lr, warmup_epochs=4, total_epochs=40, initial_lr=1e-5, peak_lr=3e-4, min_lr=1e-6):
    """Calcula la tasa de aprendizaje con calentamiento lineal y decaimiento cosenoidal suave."""
    if epoch < warmup_epochs:
        return float(initial_lr + (peak_lr - initial_lr) * (epoch + 1) / warmup_epochs)
    else:
        decay_epochs = max(1, total_epochs - warmup_epochs)
        curr = epoch - warmup_epochs
        cosine = 0.5 * (1.0 + np.cos(np.pi * curr / decay_epochs))
        return float(min_lr + (peak_lr - min_lr) * cosine)


def bloque_convolucional_residual(x, filtros, l2_factor=2e-5, nombre="bloque_res"):
    """
    Bloque convolucional con atajo de identidad y proyeccion 1x1 si cambia el numero de canales.
    Usa use_bias=False antes de Batch Normalization y regularizacion L2 aliviada.
    """
    shortcut = x
    in_filtros = x.shape[-1]
    l2_reg = tf.keras.regularizers.l2(l2_factor) if l2_factor > 0 else None
    
    y = layers.Conv2D(filtros, (3, 3), padding='same', use_bias=False,
                      kernel_regularizer=l2_reg, name=f"{nombre}_conv1")(x)
    y = layers.BatchNormalization(momentum=0.9, name=f"{nombre}_bn1")(y)
    y = layers.ReLU(name=f"{nombre}_relu1")(y)
    
    y = layers.Conv2D(filtros, (3, 3), padding='same', use_bias=False,
                      kernel_regularizer=l2_reg, name=f"{nombre}_conv2")(y)
    y = layers.BatchNormalization(momentum=0.9, name=f"{nombre}_bn2")(y)
    
    # Proyeccion de atajo 1x1 si los canales aumentan
    if in_filtros != filtros:
        shortcut = layers.Conv2D(filtros, (1, 1), padding='same', use_bias=False,
                                 kernel_regularizer=l2_reg, name=f"{nombre}_proj")(shortcut)
        shortcut = layers.BatchNormalization(momentum=0.9, name=f"{nombre}_proj_bn")(shortcut)
        
    out = layers.Add(name=f"{nombre}_add")([shortcut, y])
    return layers.ReLU(name=f"{nombre}_out")(out)


def build_tunable_cnn(
    input_shape=(384, 384, 3),
    num_classes=3,
    filter_list=[24, 48, 96, 192],       # Progresion de filtros
    initial_kernel_size=(3, 3),           # Kernel inicial del Stem
    dense_units=128,                      # Neuronas densas
    dropout_rate=0.30,                    # Regularizacion Dropout
    l2_factor=2e-5                        # Regularizacion L2 aliviada
):
    """
    Construye la arquitectura convolucional profunda con atajos residuales y GAP.
    Entrada uint8 compacta con normalizacion simetrica directa en GPU.
    """
    l2_reg = tf.keras.regularizers.l2(l2_factor) if l2_factor > 0 else None
    inputs = layers.Input(shape=input_shape, dtype='uint8', name="input_imagen")
    
    # Rescaling en GPU: uint8 [0, 255] -> float32 [-1.0, 1.0] (Gris 128 -> ~0.0)
    x = layers.Rescaling(scale=1./127.5, offset=-1.0, name="gpu_rescaling")(inputs)
    
    # Stem: Convolucion inicial y reduccion espacial 2x
    f0 = filter_list[0]
    x = layers.Conv2D(f0, initial_kernel_size, padding='same', use_bias=False,
                      kernel_regularizer=l2_reg, name="stem_conv")(x)
    x = layers.BatchNormalization(momentum=0.9, name="stem_bn")(x)
    x = layers.ReLU(name="stem_relu")(x)
    x = layers.MaxPooling2D((2, 2), name="stem_pool")(x) # -> 192x192
    
    # Etapa 1: Bloque con Atajo Residual + Reduccion
    x = bloque_convolucional_residual(x, f0, l2_factor=l2_factor, nombre="etapa1")
    x = layers.MaxPooling2D((2, 2), name="pool_e1")(x)   # -> 96x96
    
    # Etapa 2: Bloque con Atajo Residual + Reduccion
    f1 = filter_list[1]
    x = bloque_convolucional_residual(x, f1, l2_factor=l2_factor, nombre="etapa2")
    x = layers.MaxPooling2D((2, 2), name="pool_e2")(x)   # -> 48x48
    
    # Etapa 3: Bloque con Atajo Residual + Reduccion
    f2 = filter_list[2]
    x = bloque_convolucional_residual(x, f2, l2_factor=l2_factor, nombre="etapa3")
    x = layers.MaxPooling2D((2, 2), name="pool_e3")(x)   # -> 24x24
    
    # Etapa 4: Bloque con Atajo Residual (enriquecimiento final sin reduccion de dimension)
    f3 = filter_list[3]
    x = bloque_convolucional_residual(x, f3, l2_factor=l2_factor, nombre="etapa4") # -> 24x24
    
    # Cuello de botella: Global Average Pooling estandar (sin Dual Pooling)
    x = layers.GlobalAveragePooling2D(name="gap")(x) # -> (batch, f3)
    
    # Cabeza densa regularizada
    if isinstance(dense_units, list):
        for j, u in enumerate(dense_units):
            x = layers.Dense(u, use_bias=False, kernel_regularizer=l2_reg, name=f"dense_{j+1}")(x)
            x = layers.BatchNormalization(momentum=0.9, name=f"bn_densa_{j+1}")(x)
            x = layers.ReLU(name=f"relu_densa_{j+1}")(x)
            x = layers.Dropout(dropout_rate, name=f"dropout_{j+1}")(x)
    else:
        x = layers.Dense(dense_units, use_bias=False, kernel_regularizer=l2_reg, name="dense_oculta")(x)
        x = layers.BatchNormalization(momentum=0.9, name="bn_densa")(x)
        x = layers.ReLU(name="relu_densa")(x)
        x = layers.Dropout(dropout_rate, name="dropout_densa")(x)
        
    outputs = layers.Dense(num_classes, activation='softmax', dtype='float32', name="salida_softmax")(x)
    return models.Model(inputs=inputs, outputs=outputs, name="CNN_Atajos_Residuales_Custom")


def run_isolated_trial(
    nombre_exp,
    filter_list=[24, 48, 96, 192],
    initial_kernel_size=(3, 3),
    dense_units=128,
    dropout_rate=0.30,
    l2_factor=2e-5,
    optimizer='adamw',
    learning_rate=3e-4,
    epochs=40,
    patience=12,
    save_model_path=None,
    verbose_fit=1
):
    """Ejecuta un experimento de hiperparametros con aislamiento de grafo en GPU y schedule Cosine Warmup."""
    tf.keras.backend.clear_session()
    gc.collect()

    print(f"\n{'='*75}")
    print(f"  EJECUTANDO TRIAL: {nombre_exp}")
    print(f"  Filtros: {filter_list} | Kernel: {initial_kernel_size} | Densa: {dense_units} | Drop: {dropout_rate}")
    print(f"  Opt: {optimizer} (LR: {learning_rate}) | L2: {l2_factor} | Epocas: {epochs} (patience={patience})")
    print(f"{'='*75}")

    num_classes = len(class_names)
    model = build_tunable_cnn(
        input_shape=(384, 384, 3),
        num_classes=num_classes,
        filter_list=filter_list,
        initial_kernel_size=initial_kernel_size,
        dense_units=dense_units,
        dropout_rate=dropout_rate,
        l2_factor=l2_factor
    )

    opt_name = str(optimizer).lower()
    if opt_name == 'rmsprop':
        opt = tf.keras.optimizers.RMSprop(learning_rate=learning_rate, momentum=0.9)
    elif opt_name == 'nadam':
        opt = tf.keras.optimizers.Nadam(learning_rate=learning_rate)
    elif opt_name == 'adamw':
        opt = tf.keras.optimizers.AdamW(learning_rate=learning_rate, weight_decay=l2_factor if l2_factor > 0 else 1.5e-3)
    else:
        opt = tf.keras.optimizers.Adam(learning_rate=learning_rate)

    # Label smoothing calibrado a 0.03
    model.compile(
        optimizer=opt,
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.06),
        metrics=[
            'accuracy',
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall'),
            F1ScoreMetric(name='f1_score')
        ]
    )

    ckpt_trial_path = DIR_CHECKPOINTS / f"ckpt_{nombre_exp}.keras"

    # Callbacks: Warmup lineal + Cosine Decay schedule adaptativo
    lr_cb = callbacks.LearningRateScheduler(
        lambda ep, lr: schedule_cosine_warmup(ep, lr, warmup_epochs=4, total_epochs=epochs, initial_lr=1e-5, peak_lr=learning_rate, min_lr=1e-6),
        verbose=0
    )

    callbacks_trial = [
        lr_cb,
        callbacks.EarlyStopping(monitor='val_loss', patience=patience, restore_best_weights=True, verbose=0),
        callbacks.ModelCheckpoint(filepath=str(ckpt_trial_path), monitor='val_loss', save_best_only=True, verbose=0)
    ]

    t_start = time.time()
    hist = model.fit(
        train_generator,
        validation_data=val_generator,
        epochs=epochs,
        callbacks=callbacks_trial,
        verbose=verbose_fit
    )
    tiempo_total = time.time() - t_start

    # Cargar el checkpoint de menor val_loss
    if ckpt_trial_path.exists():
        try:
            model = tf.keras.models.load_model(str(ckpt_trial_path), custom_objects={'F1ScoreMetric': F1ScoreMetric, 'AdamW': tf.keras.optimizers.AdamW}, compile=False)
        except Exception:
            pass

    # Evaluacion en validacion
    preds_val = model.predict(val_generator, verbose=0)
    y_pred_idx = np.argmax(preds_val, axis=1)
    y_true_idx = np.argmax(y_val, axis=1)

    f1_macro_val = float(f1_score(y_true_idx, y_pred_idx, average='macro', zero_division=0) * 100)
    precision_val = float(precision_score(y_true_idx, y_pred_idx, average='macro', zero_division=0) * 100)
    recall_val = float(recall_score(y_true_idx, y_pred_idx, average='macro', zero_division=0) * 100)
    acc_val = float((y_true_idx == y_pred_idx).mean() * 100)
    loss_val = float(min(hist.history['val_loss']))

    val_losses = [float(v) for v in hist.history['val_loss']]
    best_epoch = int(np.argmin(val_losses)) + 1

    if save_model_path:
        model.save(save_model_path)
        print(f"Modelo guardado en: {save_model_path}")

    hist_serialized = {k: [float(v) for v in vals] for k, vals in hist.history.items()}

    res = {
        'experimento': nombre_exp,
        'filtros': str(filter_list),
        'kernel_inicial': str(initial_kernel_size),
        'densa': str(dense_units),
        'dropout': dropout_rate,
        'l2_factor': l2_factor,
        'optimizador': optimizer,
        'lr_inicial': learning_rate,
        'mejor_epoca': best_epoch,
        'val_f1_macro': round(f1_macro_val, 2),
        'val_acc': round(acc_val, 2),
        'val_precision': round(precision_val, 2),
        'val_recall': round(recall_val, 2),
        'val_loss': round(loss_val, 4),
        'tiempo_seg': round(tiempo_total, 2),
        'best_checkpoint_path': str(ckpt_trial_path),
        'history': hist_serialized
    }

    print(f"  -> Resultado: Val F1 = {res['val_f1_macro']}% | Val Acc = {res['val_acc']}% | Val Loss = {res['val_loss']} (Mejor epoca: {res['mejor_epoca']} en {res['tiempo_seg']}s)")
    return res


def run_experiment(nombre_exp, **kwargs):
    """Wrapper de conveniencia con cache persistente en JSON."""
    global historial_experimentos
    for exp in historial_experimentos:
        if exp['experimento'] == nombre_exp:
            print(f"-> [CACHE] '{nombre_exp}' recuperado: Val F1 = {exp['val_f1_macro']}% | Val Acc = {exp['val_acc']}%")
            return exp

    res = run_isolated_trial(nombre_exp, **kwargs)

    cache_dict = {}
    if RUTA_CACHE_TUNING.exists():
        try:
            with open(RUTA_CACHE_TUNING, "r", encoding="utf-8") as f:
                cache_dict = json.load(f)
        except Exception:
            cache_dict = {}

    cache_dict[nombre_exp] = res
    with open(RUTA_CACHE_TUNING, "w", encoding="utf-8") as f:
        json.dump(cache_dict, f, indent=2)

    for i, exp in enumerate(historial_experimentos):
        if exp['experimento'] == nombre_exp:
            historial_experimentos[i] = res
            return res
    historial_experimentos.append(res)
    return res

print("Infraestructura modular avanzada configurada (Atajos Residuales, Warmup + Cosine Decay y AdamW).")

## Fase 18 - Etapa 1: Capacidad Convolucional con Conexiones de Atajo Residual

Evaluamos arquitecturas con bloques de atajo residual (`layers.Add` + proyeccion $1\times 1$) a resolucion $384 \times 384$ con schedule Warmup + Cosine Decay (30 epocas):
1. `Etapa1_AtajoRes_16-128_K3`: Progresion base `[16, 32, 64, 128]`, kernel inicial `(3, 3)`.
2. `Etapa1_AtajoRes_24-192_K3`: Capacidad media `[24, 48, 96, 192]`, kernel inicial `(3, 3)`.
3. `Etapa1_AtajoRes_24-192_K5`: Capacidad media con kernel amplio `(5, 5)` para titulares de texto.
4. `Etapa1_AtajoRes_32-256_K3`: Alta capacidad `[32, 64, 128, 256]`, kernel inicial `(3, 3)`.
5. `Etapa1_AtajoRes_32-256_K5`: Alta capacidad `[32, 64, 128, 256]` con kernel inicial amplio `(5, 5)` para capturar texto y fisionomia macro.

In [ ]:
# Ejecucion de la Etapa 1: Conexiones de Atajo Residual (30 epocas, Warmup + Cosine)
EPOCHS_TRIAL = 40
PATIENCE_TRIAL = 12

r1_1 = run_experiment("Etapa1_AtajoRes_16-128_K3", filter_list=[16, 32, 64, 128], initial_kernel_size=(3, 3), epochs=EPOCHS_TRIAL, patience=PATIENCE_TRIAL, verbose_fit=1)
r1_2 = run_experiment("Etapa1_AtajoRes_24-192_K3", filter_list=[24, 48, 96, 192], initial_kernel_size=(3, 3), epochs=EPOCHS_TRIAL, patience=PATIENCE_TRIAL, verbose_fit=1)
r1_3 = run_experiment("Etapa1_AtajoRes_24-192_K5", filter_list=[24, 48, 96, 192], initial_kernel_size=(5, 5), epochs=EPOCHS_TRIAL, patience=PATIENCE_TRIAL, verbose_fit=1)
r1_4 = run_experiment("Etapa1_AtajoRes_32-256_K3", filter_list=[32, 64, 128, 256], initial_kernel_size=(3, 3), epochs=EPOCHS_TRIAL, patience=PATIENCE_TRIAL, verbose_fit=1)
r1_5 = run_experiment("Etapa1_AtajoRes_32-256_K5", filter_list=[32, 64, 128, 256], initial_kernel_size=(5, 5), epochs=EPOCHS_TRIAL, patience=PATIENCE_TRIAL, verbose_fit=1)

df_etapa1 = pd.DataFrame([exp for exp in historial_experimentos if exp['experimento'].startswith("Etapa1_")])
print("\n--- Resultados Comparativos Etapa 1 (Conexiones de Atajo Residual a 384x384) ---")
columnas_mostrar_e1 = ['experimento', 'filtros', 'kernel_inicial', 'mejor_epoca', 'val_f1_macro', 'val_acc', 'val_precision', 'val_recall', 'val_loss', 'tiempo_seg']
print(df_etapa1[[c for c in columnas_mostrar_e1 if c in df_etapa1.columns]].to_string(index=False))

# Grafico comparativo de la Etapa 1
fig, ax = plt.subplots(figsize=(12, 5))
x_indices = np.arange(len(df_etapa1))
ancho = 0.35

barras_f1 = ax.bar(x_indices - ancho/2, df_etapa1['val_f1_macro'], width=ancho, label='Val F1-Score Macro (%)', color='#1f77b4', alpha=0.9)
barras_acc = ax.bar(x_indices + ancho/2, df_etapa1['val_acc'], width=ancho, label='Val Accuracy (%)', color='#2ca02c', alpha=0.9)

ax.set_ylabel('Rendimiento en Validacion (%)', fontweight='bold')
ax.set_title('Etapa 1: Comparativa de Conexiones de Atajo Residual a 384x384 (Evaluacion F1-Score)', fontweight='bold')
ax.set_xticks(x_indices)
ax.set_xticklabels(df_etapa1['experimento'], rotation=15, ha='right')
ax.set_ylim([0, 100])
ax.legend(loc='upper right')

for bar in barras_f1:
    y = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2.0, y + 1.2, f"{y:.1f}%", ha='center', va='bottom', fontsize=9, fontweight='bold', color='#0d3b66')

for bar in barras_acc:
    y = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2.0, y + 1.2, f"{y:.1f}%", ha='center', va='bottom', fontsize=9, fontweight='bold', color='#1b5e20')

plt.tight_layout()
plt.show()

## Fase 19 - Etapa 2: Cabeza Densa y Regularizacion

Tomamos los mejores filtros y kernel de la Etapa 1 y evaluamos configuraciones de la cabeza densa con `use_bias=False`, `BatchNormalization` y `Dropout`:
1. `Dense=96`, `Dropout=0.25`, `L2=2e-5` (Ligera).
2. `Dense=128`, `Dropout=0.30`, `L2=2e-5` (Equilibrada).
3. `Dense=160`, `Dropout=0.30`, `L2=2e-5` (Capacidad amplia inspirada en clasificacion fina).

In [ ]:
# Determinar los mejores hiperparametros convolucionales de la Etapa 1
mejor_etapa1 = df_etapa1.sort_values(by=['val_f1_macro', 'val_acc', 'val_loss'], ascending=[False, False, True]).iloc[0]
mejor_filtros = eval(mejor_etapa1['filtros']) if isinstance(mejor_etapa1['filtros'], str) else mejor_etapa1['filtros']
mejor_kernel = eval(mejor_etapa1['kernel_inicial']) if isinstance(mejor_etapa1['kernel_inicial'], str) else mejor_etapa1['kernel_inicial']

print("="*65)
print(f"Hiperparametros convolucionales ganadores en Etapa 1: {mejor_etapa1['experimento']}")
print(f"  * Filtros: {mejor_filtros} | Kernel: {mejor_kernel}")
print(f"  * Val F1-Score: {mejor_etapa1['val_f1_macro']}% | Val Accuracy: {mejor_etapa1['val_acc']}%")
print("="*65)

r2_1 = run_experiment("Etapa2_Dense96_Drop0.30_L2_2e-5", filter_list=mejor_filtros, initial_kernel_size=mejor_kernel, dense_units=96, dropout_rate=0.30, l2_factor=2e-5, epochs=EPOCHS_TRIAL, patience=PATIENCE_TRIAL, verbose_fit=1)
r2_2 = run_experiment("Etapa2_Dense128_Drop0.35_L2_2e-5", filter_list=mejor_filtros, initial_kernel_size=mejor_kernel, dense_units=128, dropout_rate=0.35, l2_factor=2e-5, epochs=EPOCHS_TRIAL, patience=PATIENCE_TRIAL, verbose_fit=1)
r2_3 = run_experiment("Etapa2_Dense160_Drop0.40_L2_2e-5", filter_list=mejor_filtros, initial_kernel_size=mejor_kernel, dense_units=160, dropout_rate=0.40, l2_factor=2e-5, epochs=EPOCHS_TRIAL, patience=PATIENCE_TRIAL, verbose_fit=1)

df_etapa2 = pd.DataFrame([exp for exp in historial_experimentos if exp['experimento'].startswith("Etapa2_")])
print("\n--- Resultados Comparativos Etapa 2 (Cabeza Densa y Regularizacion) ---")
columnas_mostrar_e2 = ['experimento', 'densa', 'dropout', 'l2_factor', 'mejor_epoca', 'val_f1_macro', 'val_acc', 'val_precision', 'val_recall', 'val_loss', 'tiempo_seg']
print(df_etapa2[[c for c in columnas_mostrar_e2 if c in df_etapa2.columns]].to_string(index=False))

# Curvas de convergencia de val_loss para la Etapa 2
fig, ax = plt.subplots(figsize=(10, 5))
colores_e2 = {'Etapa2_Dense96_Drop0.30_L2_2e-5': '#C44E52', 'Etapa2_Dense128_Drop0.35_L2_2e-5': '#4C72B0', 'Etapa2_Dense160_Drop0.40_L2_2e-5': '#55A868'}

for exp in historial_experimentos:
    if exp['experimento'].startswith("Etapa2_"):
        nombre = exp['experimento']
        val_losses = exp['history']['val_loss']
        ax.plot(range(1, len(val_losses) + 1), val_losses, 'o-', label=f"{nombre} (Min Loss: {exp['val_loss']}, F1: {exp['val_f1_macro']}%)", color=colores_e2.get(nombre, '#333333'))

ax.set_title('Etapa 2: Dinamica de Convergencia de Val Loss (Cabeza Densa)', fontweight='bold')
ax.set_xlabel('Epoca')
ax.set_ylabel('Validation Loss')
ax.legend()
plt.tight_layout()
plt.show()

## Fase 20 - Etapa 3: Optimizacion y Learning Rate (AdamW vs RMSprop vs Adam)

Evaluamos el desempeno de los optimizadores con schedule Warmup + Cosine Decay a 30 epocas:
1. `AdamW(learning_rate=3e-4, weight_decay=1e-4)`: Desacopla el decaimiento de pesos, protegiendo filtros de bordes finos.
2. `AdamW(learning_rate=2.5e-4, weight_decay=1e-4)`: Configuracion de convergencia suave de alta precision.
3. `RMSprop(learning_rate=7e-4)`: RMSprop adaptativo con momentum 0.9.
4. `Adam(learning_rate=5e-4)`: Adam clasico con momento de primer y segundo orden.

In [ ]:
# Determinar la mejor configuracion densa y regularizacion de la Etapa 2
mejor_etapa2 = df_etapa2.sort_values(by=['val_f1_macro', 'val_acc', 'val_loss'], ascending=[False, False, True]).iloc[0]
mejor_densa = eval(mejor_etapa2['densa']) if isinstance(mejor_etapa2['densa'], str) and '[' in mejor_etapa2['densa'] else int(mejor_etapa2['densa'])
mejor_dropout = float(mejor_etapa2['dropout'])
mejor_l2 = float(mejor_etapa2['l2_factor'])

print("="*65)
print(f"Clasificador y regularizacion ganadores en Etapa 2: {mejor_etapa2['experimento']}")
print(f"  * Neuronas: {mejor_densa} | Dropout: {mejor_dropout} | L2: {mejor_l2}")
print(f"  * Val F1-Score: {mejor_etapa2['val_f1_macro']}% | Val Accuracy: {mejor_etapa2['val_acc']}%")
print("="*65)

r3_1 = run_experiment("Etapa3_AdamW_3e-4", filter_list=mejor_filtros, initial_kernel_size=mejor_kernel, dense_units=mejor_densa, dropout_rate=mejor_dropout, l2_factor=mejor_l2, optimizer='adamw', learning_rate=3e-4, epochs=EPOCHS_TRIAL, patience=PATIENCE_TRIAL, verbose_fit=1)
r3_2 = run_experiment("Etapa3_AdamW_2.5e-4", filter_list=mejor_filtros, initial_kernel_size=mejor_kernel, dense_units=mejor_densa, dropout_rate=mejor_dropout, l2_factor=mejor_l2, optimizer='adamw', learning_rate=2.5e-4, epochs=EPOCHS_TRIAL, patience=PATIENCE_TRIAL, verbose_fit=1)
r3_3 = run_experiment("Etapa3_RMSprop_7e-4", filter_list=mejor_filtros, initial_kernel_size=mejor_kernel, dense_units=mejor_densa, dropout_rate=mejor_dropout, l2_factor=mejor_l2, optimizer='rmsprop', learning_rate=7e-4, epochs=EPOCHS_TRIAL, patience=PATIENCE_TRIAL, verbose_fit=1)
r3_4 = run_experiment("Etapa3_Adam_5e-4", filter_list=mejor_filtros, initial_kernel_size=mejor_kernel, dense_units=mejor_densa, dropout_rate=mejor_dropout, l2_factor=mejor_l2, optimizer='adam', learning_rate=5e-4, epochs=EPOCHS_TRIAL, patience=PATIENCE_TRIAL, verbose_fit=1)

df_etapa3 = pd.DataFrame([exp for exp in historial_experimentos if exp['experimento'].startswith("Etapa3_")])
print("\n--- Resultados Comparativos Etapa 3 (Optimizador y Tasa de Aprendizaje) ---")
columnas_mostrar_e3 = ['experimento', 'optimizador', 'lr_inicial', 'mejor_epoca', 'val_f1_macro', 'val_acc', 'val_precision', 'val_recall', 'val_loss', 'tiempo_seg']
print(df_etapa3[[c for c in columnas_mostrar_e3 if c in df_etapa3.columns]].to_string(index=False))

# Grafico comparativo de la Etapa 3
fig, ax = plt.subplots(figsize=(10, 5))
barras3_f1 = ax.bar(df_etapa3['experimento'], df_etapa3['val_f1_macro'], color=['#1b5e20', '#2e7d32', '#d35400', '#2980b9'], width=0.45, alpha=0.9)
ax.set_ylabel('Val F1-Score Macro (%)', fontweight='bold')
ax.set_title('Etapa 3: Comparativa de Optimizadores con Atajos Residuales (F1-Score)', fontweight='bold')
ax.set_ylim([0, 100])
ax.set_xticklabels(df_etapa3['experimento'], rotation=15, ha='right')
for bar in barras3_f1:
    y = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2.0, y + 1.5, f"{y:.2f}%", ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

## Fase 21 - Etapa 4: Evaluacion Oficial del Modelo Campeon vs Baseline

Identificamos la combinacion con mayor **F1-Score Macro** en validacion de todas las etapas, cargamos directamente sus mejores pesos preservados y contrastamos oficialmente sus metricas frente al Modelo Base.

In [ ]:
# Identificar la configuracion campeona absoluta basada en F1-Score Macro de validacion
df_todos = pd.DataFrame(historial_experimentos)
campeon_row = df_todos.sort_values(by=['val_f1_macro', 'val_acc', 'val_loss'], ascending=[False, False, True]).iloc[0]

print("="*65)
print(f"CONFIGURACION CAMPEONA SELECCIONADA POR F1-SCORE: {campeon_row['experimento']}")
print(f"Filtros: {campeon_row.get('filtros', '[24,48,96,192]')} | Kernel: {campeon_row.get('kernel_inicial', '(3,3)')} | Densa: {campeon_row.get('densa', '128')} | Drop: {campeon_row['dropout']} | Opt: {campeon_row['optimizador']} ({campeon_row['lr_inicial']})")
print(f"Desempeno alcanzado: Val F1 Macro = {campeon_row['val_f1_macro']}% | Val Accuracy = {campeon_row['val_acc']}% | Val Loss = {campeon_row['val_loss']}")
print("="*65)

ruta_modelo_campeon = Path("modelo_memes_campeon.keras")
ruta_historia_campeon = Path("historia_campeon.json")

# 1. Recuperar directamente el mejor checkpoint guardado durante el trial ganador
best_ckpt_str = campeon_row.get('best_checkpoint_path', '')
best_ckpt_path = Path(best_ckpt_str) if best_ckpt_str else None

if best_ckpt_path and best_ckpt_path.exists():
    print(f"\nCargando checkpoint optimo del trial ganador desde: {best_ckpt_path.resolve()}")
    modelo_campeon = tf.keras.models.load_model(str(best_ckpt_path), custom_objects={'F1ScoreMetric': F1ScoreMetric, 'AdamW': tf.keras.optimizers.AdamW}, compile=False)
    modelo_campeon.save(ruta_modelo_campeon)
    hist_campeon_dict = campeon_row['history']
    with open(ruta_historia_campeon, "w", encoding="utf-8") as f:
        json.dump(hist_campeon_dict, f)
    print(f"Modelo Campeon persistido exitosamente en: {ruta_modelo_campeon.resolve()}")
elif ruta_modelo_campeon.exists() and ruta_historia_campeon.exists():
    print(f"\nCargando Modelo Campeon previamente entrenado desde: {ruta_modelo_campeon.resolve()}")
    modelo_campeon = tf.keras.models.load_model(ruta_modelo_campeon, custom_objects={'F1ScoreMetric': F1ScoreMetric, 'AdamW': tf.keras.optimizers.AdamW}, compile=False)
    with open(ruta_historia_campeon, "r", encoding="utf-8") as f:
        hist_campeon_dict = json.load(f)
else:
    print("\nEntrenando Modelo Campeon definitivo con guardado persistente...")
    filtros_c = eval(campeon_row['filtros']) if isinstance(campeon_row.get('filtros'), str) else [24, 48, 96, 192]
    kernel_c = eval(campeon_row['kernel_inicial']) if isinstance(campeon_row.get('kernel_inicial'), str) else (3, 3)
    densa_c = eval(campeon_row['densa']) if isinstance(campeon_row.get('densa'), str) and '[' in campeon_row['densa'] else int(campeon_row.get('densa', 128))
    
    res_campeon = run_isolated_trial(
        nombre_exp="Modelo_Campeon_Definitivo",
        filter_list=filtros_c,
        initial_kernel_size=kernel_c,
        dense_units=densa_c,
        dropout_rate=float(campeon_row['dropout']),
        l2_factor=float(campeon_row.get('l2_factor', 2e-5)),
        optimizer=str(campeon_row['optimizador']),
        learning_rate=float(campeon_row['lr_inicial']),
        epochs=40,
        patience=15,
        save_model_path=str(ruta_modelo_campeon),
        verbose_fit=1
    )
    hist_campeon_dict = res_campeon['history']
    with open(ruta_historia_campeon, "w", encoding="utf-8") as f:
        json.dump(hist_campeon_dict, f)
    modelo_campeon = tf.keras.models.load_model(str(ruta_modelo_campeon), custom_objects={'F1ScoreMetric': F1ScoreMetric, 'AdamW': tf.keras.optimizers.AdamW}, compile=False)

# Evaluacion de predicciones en validacion
y_reales_c = []
y_pred_c = []

for idx_b in range(len(val_generator)):
    bx, by = val_generator[idx_b]
    preds = modelo_campeon.predict(bx, verbose=0)
    y_reales_c.extend(np.argmax(by, axis=1))
    y_pred_c.extend(np.argmax(preds, axis=1))

y_reales_c = np.array(y_reales_c)
y_pred_c = np.array(y_pred_c)
labels_idx = list(range(len(class_names)))

print("\nReporte Oficial de Clasificacion - Modelo Campeon:")
print(classification_report(y_reales_c, y_pred_c, labels=labels_idx, target_names=class_names, zero_division=0))

# Visualizacion comparativa: Curvas, Matriz de Confusion y Tabla Resumen
fig = plt.figure(figsize=(16, 10))

# 1. Curva de Accuracy
ax_acc = plt.subplot(2, 2, 1)
ax_acc.plot(hist_campeon_dict['accuracy'], 'o-', label='Train Accuracy', color='#1f77b4', linewidth=2)
ax_acc.plot(hist_campeon_dict['val_accuracy'], 's--', label='Val Accuracy', color='#2ca02c', linewidth=2)
ax_acc.set_title('Curva de Precision - Modelo Campeon (Con Atajos Residuales)', fontweight='bold')
ax_acc.set_xlabel('Epoca')
ax_acc.set_ylabel('Accuracy')
ax_acc.grid(True, linestyle='--', alpha=0.5)
ax_acc.legend()

# 2. Curva de Loss
ax_loss = plt.subplot(2, 2, 2)
ax_loss.plot(hist_campeon_dict['loss'], 'o-', label='Train Loss', color='#d62728', linewidth=2)
ax_loss.plot(hist_campeon_dict['val_loss'], 's--', label='Val Loss', color='#9467bd', linewidth=2)
ax_loss.set_title('Curva de Perdida - Modelo Campeon', fontweight='bold')
ax_loss.set_xlabel('Epoca')
ax_loss.set_ylabel('Loss')
ax_loss.grid(True, linestyle='--', alpha=0.5)
ax_loss.legend()

# 3. Matriz de Confusion
ax_cm = plt.subplot(2, 2, 3)
cm_c = confusion_matrix(y_reales_c, y_pred_c, labels=labels_idx)
sns.heatmap(cm_c, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, cbar=False, ax=ax_cm)
ax_cm.set_title('Matriz de Confusion - Modelo Campeon Definitivo', fontweight='bold')
ax_cm.set_xlabel('Predicho', fontweight='bold')
ax_cm.set_ylabel('Real', fontweight='bold')

# 4. Tabla Comparativa Resumen
ax_tbl = plt.subplot(2, 2, 4)
ax_tbl.axis('off')
acc_final_c = max(hist_campeon_dict['val_accuracy']) * 100
loss_final_c = min(hist_campeon_dict['val_loss'])

tabla_datos = [
    ["Metrica / Atributo", "Baseline Original (Parte I)", "Modelo Campeon (Atajos Residuales)"],
    ["Val Accuracy", "67.33%", f"{acc_final_c:.2f}%"],
    ["Val Loss", "0.9064", f"{loss_final_c:.4f}"],
    ["Arquitectura", "Simple Conv (4 capas planas)", "Atajos Residuales (8 capas + Proyeccion)"],
    ["Resolucion", "224 x 224", "384 x 384 (+194% px)"],
    ["Transferencia PCIe", "float32 (12 MB/lote)", "uint8 (3 MB/lote, 4x mas rapido)"],
    ["Schedule LR", "ReduceLROnPlateau", "Warmup Lineal + Cosine Decay"],
    ["Optimizador", "Adam (5e-4)", f"{campeon_row['optimizador']} ({campeon_row['lr_inicial']})"]
]
tabla = ax_tbl.table(cellText=tabla_datos, loc='center', cellLoc='center')
tabla.auto_set_font_size(False)
tabla.set_fontsize(9.5)
tabla.scale(1.2, 1.6)
ax_tbl.set_title('Tabla Comparativa Oficial: Baseline vs Campeon', fontweight='bold')

plt.tight_layout()
plt.show()

## Conclusiones de la Busqueda Sistematica con Conexiones de Atajo Residual (384x384)

1. **Resolucion de Alta Fidelidad ($384 \times 384$ px)**: Preserva el texto original y rasgos faciales sin distorsion por Letterbox, proporcionando mapas de activacion de $24 \times 24$ para el `GlobalAveragePooling2D`.
2. **Conexiones de Atajo Residual (`layers.Add`)**: Superan la barrera del subajuste al permitir que los gradientes retropropaguen sin desvanecimiento a traves de las 8 capas convolucionales.
3. **Eficiencia y Estabilidad**: El uso de `use_bias=False` antes de `BatchNormalization`, la transferencia `uint8` y el schedule de Warmup Lineal con Decaimiento Cosenoidal garantizan convergencia monotona en la GPU GPU Acelerada (NVIDIA T4 / Colab) sin cuellos de botella.

## Fase 22: Explicabilidad Visual con Grad-CAM (Gradient-weighted Class Activation Mapping)

**Grad-CAM** permite auditar el modelo para verificar qué regiones específicas de cada meme activaron con mayor fuerza los mapas de características finales antes del Global Average Pooling.

### Objetivos de la Auditoría XAI:
1. **Garantizar ausencia de 'Shortcut Learning'**: Comprobar que el modelo no esté basando sus predicciones en las barras de relleno gris, bordes o marcas de agua, sino en la semántica del meme.
2. **Foco por Clase**:
   * En **animales**: Debe concentrarse en hocicos, ojos, pelaje o rasgos anatómicos del animal.
   * En **animado**: Debe concentrarse en trazos vectoriales, estética de caricatura u ojos de animación.
   * En **personas**: Debe concentrarse en rostros humanos, expresiones faciales y gestos.

*Nota*: No requiere reentrenamiento; opera directamente extrayendo los gradientes del modelo entrenado.

In [ ]:
# Implementación de Grad-CAM nativo con tf.GradientTape
def make_gradcam_heatmap(img_array, model, last_conv_layer_name="etapa4_out", pred_index=None):
    """Calcula el mapa de calor Grad-CAM para una imagen y una capa convolucional objetivo."""
    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0.0)
    max_val = tf.math.reduce_max(heatmap)
    if max_val > 0:
        heatmap = heatmap / max_val
    return heatmap.numpy()

def superimpose_gradcam(img_uint8, heatmap, alpha=0.45):
    """Superpone el mapa de calor sobre la imagen original usando mapa de colores JET."""
    heatmap_resized = cv2.resize(heatmap, (img_uint8.shape[1], img_uint8.shape[0]))
    heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
    heatmap_rgb = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
    superimposed = np.uint8(heatmap_rgb * alpha + img_uint8 * (1 - alpha))
    return superimposed, heatmap_resized

# Seleccionar el mejor modelo disponible (campeon o baseline)
modelo_eval_xai = modelo_campeon if ('modelo_campeon' in locals() and modelo_campeon is not None) else modelo_cnn
clases_nombres = list(etiquetas_a_idx.keys())

# Seleccionar una muestra representativa de cada clase
muestras_xai = []
for c_idx, c_nom in enumerate(clases_nombres):
    idxs = np.where(y_val_idx == c_idx)[0]
    if len(idxs) > 0:
        muestras_xai.append((idxs[0], c_nom, c_idx))

# Localizar la última capa convolucional disponible
capas_conv = [l.name for l in modelo_eval_xai.layers if 'out' in l.name or 'conv' in l.name or 'add' in l.name]
target_layer = "etapa4_out" if "etapa4_out" in [l.name for l in modelo_eval_xai.layers] else capas_conv[-1]
print(f"Capa convolucional seleccionada para Grad-CAM: {target_layer}")

fig, axes = plt.subplots(len(muestras_xai), 3, figsize=(15, 5 * len(muestras_xai)))
if len(muestras_xai) == 1:
    axes = np.expand_dims(axes, 0)

for row, (idx_muestra, real_nom, real_idx) in enumerate(muestras_xai):
    img_uint8 = X_val_raw[idx_muestra]
    img_input = np.expand_dims(img_uint8, axis=0)
    
    preds = modelo_eval_xai.predict(img_input, verbose=0)[0]
    pred_idx = np.argmax(preds)
    pred_nom = clases_nombres[pred_idx]
    conf = preds[pred_idx] * 100
    
    heatmap = make_gradcam_heatmap(img_input, modelo_eval_xai, last_conv_layer_name=target_layer, pred_index=pred_idx)
    gradcam_img, heatmap_resized = superimpose_gradcam(img_uint8, heatmap)
    
    # 1. Imagen Original
    axes[row, 0].imshow(img_uint8)
    axes[row, 0].set_title(f"Clase Real: {real_nom.upper()}\nPredicción: {pred_nom.upper()} ({conf:.1f}%)", fontsize=11, fontweight='bold')
    axes[row, 0].axis('off')
    
    # 2. Mapa de Calor Grad-CAM
    im1 = axes[row, 1].imshow(heatmap_resized, cmap='jet')
    axes[row, 1].set_title(f"Grad-CAM Heatmap\n({target_layer})", fontsize=11)
    axes[row, 1].axis('off')
    plt.colorbar(im1, ax=axes[row, 1], fraction=0.046, pad=0.04)
    
    # 3. Superposición
    axes[row, 2].imshow(gradcam_img)
    axes[row, 2].set_title(f"Superposición Grad-CAM\n(Región de Atención de la CNN)", fontsize=11, fontweight='bold')
    axes[row, 2].axis('off')

plt.suptitle("Auditoría de Explicabilidad Visual Grad-CAM (Modelo de Clasificación de Memes)", fontsize=15, fontweight='bold', y=0.99)
plt.tight_layout()
plt.show()

## Fase 23: Explicabilidad Local con LIME (Local Interpretable Model-agnostic Explanations)

**LIME** segmenta la imagen en superpíxeles y perturba regiones para evaluar la sensibilidad de la predicción, mostrando en verde las áreas que soportan la clase ganadora.

In [ ]:
# Implementación de LIME para clasificación de imágenes
try:
    from lime import lime_image
    from skimage.segmentation import mark_boundaries
    LIME_ACTIVO = True
except ImportError:
    print("Aviso: Librería 'lime' no instalada. Instálala ejecutando: !pip install lime")
    LIME_ACTIVO = False

if LIME_ACTIVO:
    explainer = lime_image.LimeImageExplainer()

    def predict_para_lime(images):
        imgs_uint8 = np.clip(images, 0, 255).astype('uint8')
        return modelo_eval_xai.predict(imgs_uint8, verbose=0)

    idx_lime, nombre_clase_lime, _ = muestras_xai[0]
    img_muestra = X_val_raw[idx_lime]
    
    print(f"Calculando explicación LIME para una muestra de '{nombre_clase_lime}' (500 perturbaciones)...")
    explanation = explainer.explain_instance(
        img_muestra.astype('double'),
        predict_para_lime,
        top_labels=3,
        hide_color=128,
        num_samples=500
    )
    
    top_pred = explanation.top_labels[0]
    temp, mask = explanation.get_image_and_mask(
        top_pred,
        positive_only=False,
        num_features=5,
        hide_rest=False
    )
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(img_muestra)
    axes[0].set_title(f"Imagen Original ({nombre_clase_lime})", fontsize=12)
    axes[0].axis('off')
    
    axes[1].imshow(mark_boundaries(temp / 255.0, mask))
    axes[1].set_title(f"Explicación LIME (Clase Predicha: {clases_nombres[top_pred]})\nSuperpíxeles con Mayor Evidencia", fontsize=12, fontweight='bold')
    axes[1].axis('off')
    
    plt.suptitle("Auditoría LIME de Superpíxeles Explicativos", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("Ejecuta '!pip install lime' para visualizar las explicaciones con LIME.")